# LibBOT — BM25 vs MiniLM (prefix / windowed) vs hybrid retrieval on SciFact

This is the **one shared notebook** for the whole pipeline. Each member owns one section. Sections talk to each other **only through files on disk** (see the *Data contract* below), so you can start from your own section without re-running everyone else's.

| § | Section | Owner | Status |
|---|---|---|---|
| 1 | Setup, config, shared helpers, data contract | Member 1 (scaffolded by M2) | ✅ implemented |
| 2 | Dataset loading + claim→document→evidence mapping | **Member 2** | ✅ implemented |
| A | Checkpoint A: load shared data | shared | ✅ implemented |
| 3 | BM25 baseline | Member 3 | ✅ implemented |
| 4 | Prefix MiniLM | Member 4 | ✅ implemented |
| 5 | Windowed MiniLM | Member 5 | ✅ implemented |
| 6 | Truncation / evidence-retention diagnostic | **Members 2 + 5** | ✅ implemented |
| 7 | Hybrid fusion + α tuning | Member 6 | ✅ implemented |
| 8 | Final evaluation (quality + efficiency) | Member 1 | ✅ implemented |
| 9 | Statistical comparison, results table, charts | Member 6 | ✅ implemented |
| 10 | Demo export + reproducibility log | Member 6 / Member 1 | ✅ implemented |

## Working rules (read once)

1. **Edit only your own section.** Changes to Section 1 (config/helpers) need a heads-up to the group, because everyone depends on them.
2. **Inputs and outputs go through disk.** Read from `data/processed/` and write your rankings with `save_run()` into `results/runs/`. Never pass data between sections via a variable that only exists in your own session.
3. **One ranking format for everyone.** Use `scores_to_run()` + `save_run()`. `save_run()` validates the format and refuses bad files.
4. **No test leakage.** α tuning, window settings and any other choice uses **train** only. Test qrels and evidence locations are only for final evaluation and the diagnostic.
5. **Git hygiene for a shared notebook:** run `nbstripout --install` once after cloning (strips outputs on commit), always `git pull` before you start, and commit only changes inside your section. If two people must edit at once, work on branches and merge one at a time.
6. **Run order:** Section 1 → (Section 2 *or* Checkpoint A) → your section.

---
# 1 · Setup, config and shared helpers  *(Member 1 — scaffolded by Member 2)*

In [ ]:
# Uncomment on a fresh machine / Colab.
# On Colab, first:  !git clone https://github.com/yapzukai/libbot.git  then  %cd libbot
# %pip install -q -r requirements.txt

In [ ]:
import os, sys, re, json, time, random, platform, hashlib, tarfile, zipfile, shutil, urllib.request, warnings
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 160)

# ---------------- Global config (team-wide; change only by agreement) ----------------
SEED        = 42
K           = 10        # evaluation cutoff: Recall@10, nDCG@10
RUN_DEPTH   = 1000      # docs saved per query in every run file (fusion needs more than the top-10)
MODEL_NAME  = "sentence-transformers/all-MiniLM-L6-v2"
MAX_SEQ_LEN = 256       # MiniLM input limit in word-pieces, INCLUDING [CLS] and [SEP]
DOC_SEP     = " "       # doc_text = (title + DOC_SEP + text).strip()  -> the ONE string every method encodes/indexes
ALPHA_GRID  = [round(a, 2) for a in np.linspace(0.0, 1.0, 11)]   # same grid for both hybrids
METHODS     = ["bm25", "prefix_minilm", "window_minilm", "hybrid_prefix", "hybrid_window"]
SPLITS      = ["train", "test"]   # BEIR splits. train = tuning, test = final (300 queries)

# ---------------- Paths ----------------
ROOT        = Path(os.environ.get("LIBBOT_ROOT", Path.cwd())).resolve()
RAW_DIR     = ROOT / "data" / "raw"
PROC_DIR    = ROOT / "data" / "processed"
CACHE_DIR   = ROOT / "cache"
RESULTS_DIR = ROOT / "results"
RUNS_DIR    = RESULTS_DIR / "runs"
METRICS_DIR = RESULTS_DIR / "metrics"
EFF_DIR     = RESULTS_DIR / "efficiency"
FIG_DIR     = RESULTS_DIR / "figures"
for d in [RAW_DIR, PROC_DIR, CACHE_DIR, RUNS_DIR, METRICS_DIR, EFF_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)

def environment_info():
    info = {
        "timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "python": sys.version.split()[0],
        "platform": platform.platform(),
        "processor": platform.processor() or platform.machine(),
        "cpu_count": os.cpu_count(),
        "numpy": np.__version__, "pandas": pd.__version__,
    }
    try:
        import torch
        info["torch"] = torch.__version__
        info["cuda"] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
        info["mps"] = bool(getattr(torch.backends, "mps", None) and torch.backends.mps.is_available())
    except ImportError:
        info["torch"] = None
    for lib in ["transformers", "sentence_transformers", "rank_bm25"]:
        try:
            info[lib] = __import__(lib).__version__
        except Exception:
            info[lib] = None
    return info

ENV = environment_info()
CONFIG = dict(SEED=SEED, K=K, RUN_DEPTH=RUN_DEPTH, MODEL_NAME=MODEL_NAME, MAX_SEQ_LEN=MAX_SEQ_LEN,
              DOC_SEP=DOC_SEP, ALPHA_GRID=ALPHA_GRID, METHODS=METHODS, SPLITS=SPLITS)
(RESULTS_DIR / "config_snapshot.json").write_text(json.dumps({"config": CONFIG, "environment": ENV}, indent=2))
print("ROOT:", ROOT)
pd.Series(ENV)

### Data contract (the shared format)

All IDs are **strings** (`doc_id`, `query_id`). `doc_text` is the single canonical document string; character offsets below are offsets into `doc_text`.

**`data/processed/`** — written by Section 2

| File | One row per | Key columns |
|---|---|---|
| `corpus.parquet` | document (5,183) | `doc_id, title, text, doc_text, sentences[list], sent_start[list], sent_end[list], n_sentences, n_words, structured` |
| `queries.parquet` | claim (1,109) | `query_id, claim, split ∈ {train,test,unused}, orig_split ∈ {train,dev}` |
| `qrels.parquet` | (query, relevant doc) | `query_id, doc_id, relevance, split` |
| `qrel_pairs.parquet` | (query, relevant doc) + evidence | qrels columns + `claim, has_evidence, label, evidence_sentence_ids[list], ev_char_start[list], ev_char_end[list], first_evidence_char, rel_position_first, …` |
| `evidence_sentences.parquet` | evidence sentence | `query_id, doc_id, split, label, rationale_set, sentence_idx, char_start, char_end, sentence_text` |
| `manifest.json` | — | counts, file hashes, data source, timestamp |

Written by Section 6.1: `doc_token_stats.parquet` (per doc: word-piece count, retained character span) and `prefix_retention.parquet` (per qrel pair: complete / partial / absent).

**`results/runs/<method>__<split>.parquet`** — one per method × split, written with `save_run()`

| column | type | meaning |
|---|---|---|
| `query_id` | str | every evaluation query of that split |
| `doc_id` | str | must exist in `corpus.parquet` |
| `score` | float | higher = more relevant (raw score; fusion normalises later) |
| `rank` | int | 1 … `RUN_DEPTH`, contiguous, sorted by score desc (ties → doc_id asc) |

`<method>` ∈ `bm25, prefix_minilm, window_minilm, hybrid_prefix, hybrid_window`.

**`results/efficiency/efficiency_log.jsonl`** — append with `log_efficiency(method, ...)`: `indexing_time_s, n_embeddings, storage_bytes, mean_latency_ms, p95_latency_ms, ...` plus hardware info automatically.

In [ ]:
# ---------------- Shared helpers (format + validation). Everyone uses these. ----------------

def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def write_jsonl(rows, path):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

def norm_ws(s):
    """Collapse all whitespace runs to one space (for robust text comparison)."""
    return re.sub(r"\s+", " ", str(s)).strip()

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

CHECK_LOG = []
def check(name, ok, detail="", hard=False):
    """Record a sanity check. hard=True raises on failure; otherwise it is a warning."""
    ok = bool(ok)
    status = "PASS" if ok else ("FAIL" if hard else "WARN")
    CHECK_LOG.append({"check": name, "status": status, "detail": str(detail)})
    print(f"[{status}] {name}" + (f"  — {detail}" if detail else ""))
    if hard and not ok:
        raise AssertionError(f"{name}: {detail}")
    return ok

# ---- processed data registry ----
PROCESSED_FILES = {
    "corpus":             PROC_DIR / "corpus.parquet",
    "queries":            PROC_DIR / "queries.parquet",
    "qrels":              PROC_DIR / "qrels.parquet",
    "qrel_pairs":         PROC_DIR / "qrel_pairs.parquet",
    "evidence_sentences": PROC_DIR / "evidence_sentences.parquet",
    "doc_token_stats":    PROC_DIR / "doc_token_stats.parquet",
    "prefix_retention":   PROC_DIR / "prefix_retention.parquet",
}

def _restore_lists(df):
    # parquet round-trips list columns as numpy arrays; turn them back into python lists
    for c in df.columns:
        if len(df) and isinstance(df[c].iloc[0], np.ndarray):
            df[c] = df[c].apply(list)
    return df

def load_processed(*names):
    """load_processed('corpus', 'qrels') -> tuple of DataFrames (single DataFrame if one name)."""
    out = []
    for n in names:
        p = PROCESSED_FILES[n]
        if not p.exists():
            raise FileNotFoundError(f"{p} not found — run Section 2 (or pull data/processed/ from the repo).")
        out.append(_restore_lists(pd.read_parquet(p)))
    return out[0] if len(out) == 1 else tuple(out)

# ---- evaluation-query helpers ----
def eval_queries(split, queries_df=None):
    """Queries that have at least one qrel in `split` (BEIR convention: only these are evaluated)."""
    q = queries_df if queries_df is not None else load_processed("queries")
    return q[q["split"] == split].reset_index(drop=True)

def qrels_dict(split, qrels_df=None):
    """{query_id: {doc_id: relevance}} — the format pytrec_eval / BEIR evaluators expect."""
    qr = qrels_df if qrels_df is not None else load_processed("qrels")
    qr = qr[qr["split"] == split]
    out = {}
    for qid, did, rel in qr[["query_id", "doc_id", "relevance"]].itertuples(index=False):
        out.setdefault(qid, {})[did] = int(rel)
    return out

# ---- run (ranking) format ----
RUN_COLUMNS = ["query_id", "doc_id", "score", "rank"]

def run_path(method, split):
    return RUNS_DIR / f"{method}__{split}.parquet"

def scores_to_run(scores, query_ids, doc_ids, depth=RUN_DEPTH):
    """Turn a (n_queries x n_docs) score matrix (higher = better) into the shared run format."""
    scores = np.asarray(scores, dtype=np.float64)
    assert scores.shape == (len(query_ids), len(doc_ids)), f"scores {scores.shape} vs ({len(query_ids)}, {len(doc_ids)})"
    depth = min(depth, scores.shape[1])
    doc_ids = np.asarray([str(d) for d in doc_ids], dtype=object)
    parts = []
    for i, qid in enumerate(query_ids):
        s = scores[i]
        top = np.argpartition(-s, depth - 1)[:depth] if depth < len(s) else np.arange(len(s))
        parts.append(pd.DataFrame({"query_id": str(qid), "doc_id": doc_ids[top], "score": s[top]}))
    run = pd.concat(parts, ignore_index=True)
    return _rank_run(run)

def _rank_run(run):
    run = run.sort_values(["query_id", "score", "doc_id"], ascending=[True, False, True], kind="mergesort")
    run["rank"] = run.groupby("query_id").cumcount() + 1
    return run.reset_index(drop=True)[RUN_COLUMNS]

_CORPUS_IDS = None
def _corpus_ids():
    global _CORPUS_IDS
    if _CORPUS_IDS is None and PROCESSED_FILES["corpus"].exists():
        _CORPUS_IDS = set(pd.read_parquet(PROCESSED_FILES["corpus"], columns=["doc_id"])["doc_id"])
    return _CORPUS_IDS

def validate_run(run, split=None, depth=RUN_DEPTH, name="run"):
    """Raises AssertionError if `run` breaks the shared format."""
    missing = [c for c in RUN_COLUMNS if c not in run.columns]
    assert not missing, f"{name}: missing columns {missing}"
    assert run["query_id"].map(type).eq(str).all() and run["doc_id"].map(type).eq(str).all(), f"{name}: ids must be str"
    assert np.isfinite(run["score"].to_numpy(dtype=float)).all(), f"{name}: non-finite scores"
    assert not run.duplicated(["query_id", "doc_id"]).any(), f"{name}: duplicate (query_id, doc_id) rows"
    g = run.sort_values(["query_id", "rank"]).groupby("query_id")
    sizes = g.size()
    assert (sizes <= depth).all(), f"{name}: more than {depth} docs for some query"
    ranks_ok = g["rank"].apply(lambda r: (r.to_numpy() == np.arange(1, len(r) + 1)).all())
    assert ranks_ok.all(), f"{name}: ranks must be 1..n per query; bad queries e.g. {list(ranks_ok[~ranks_ok].index[:3])}"
    mono = g["score"].apply(lambda s: bool((np.diff(s.to_numpy()) <= 1e-12).all()))
    assert mono.all(), f"{name}: scores must be non-increasing with rank"
    cids = _corpus_ids()
    if cids is not None:
        bad = set(run["doc_id"]) - cids
        assert not bad, f"{name}: {len(bad)} doc_ids not in corpus, e.g. {list(bad)[:3]}"
    if split is not None and PROCESSED_FILES["queries"].exists():
        expected = set(eval_queries(split)["query_id"])
        got = set(run["query_id"])
        assert got == expected, f"{name}: query set mismatch for split={split} (missing {len(expected-got)}, extra {len(got-expected)})"
    return True

def save_run(run, method, split, depth=RUN_DEPTH):
    assert method in METHODS, f"method must be one of {METHODS}"
    assert split in SPLITS, f"split must be one of {SPLITS}"
    run = run[RUN_COLUMNS].copy()
    run["query_id"] = run["query_id"].astype(str)
    run["doc_id"] = run["doc_id"].astype(str)
    run["rank"] = run["rank"].astype(int)
    validate_run(run, split=split, depth=depth, name=f"{method}/{split}")
    path = run_path(method, split)
    run.to_parquet(path, index=False)
    print(f"saved {path.relative_to(ROOT)}  ({run['query_id'].nunique()} queries, {len(run):,} rows)")
    return path

def load_run(method, split):
    return pd.read_parquet(run_path(method, split))

def run_status():
    rows = []
    for m in METHODS:
        for s in SPLITS:
            p = run_path(m, s)
            rows.append({"method": m, "split": s, "exists": p.exists(),
                         "rows": len(pd.read_parquet(p, columns=["rank"])) if p.exists() else 0})
    return pd.DataFrame(rows).pivot(index="method", columns="split", values="exists").reindex(METHODS)

def log_efficiency(method, **metrics):
    """Append one efficiency record, e.g. log_efficiency('bm25', indexing_time_s=1.2, mean_latency_ms=9.0)."""
    rec = {"method": method, "timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
           "hardware": f"{ENV['processor']} | cuda={ENV.get('cuda')} | mps={ENV.get('mps')}", **metrics}
    with open(EFF_DIR / "efficiency_log.jsonl", "a", encoding="utf-8") as f:
        f.write(json.dumps(rec) + "\n")
    return rec

print("helpers ready")

---
# 2 · Dataset loading and evidence mapping  *(Member 2)*

**Goal.** Load both versions of SciFact and join them into one clean set of tables:

- **BEIR SciFact** gives the retrieval task: corpus, queries (claims) and relevance labels (qrels, train/test).
- **Original SciFact** gives what BEIR drops: the abstract split into sentences, and for every claim–document pair the **expert-annotated evidence sentences** and label (SUPPORT / CONTRADICT).

The output is a mapping **claim → relevant document → relevance label → evidence sentences (with character positions inside `doc_text`)**. Section 6 uses those positions to check whether evidence falls past MiniLM's 256-word-piece cutoff.

**One split detail matters a lot.** BEIR's *test* qrels (300 claims) are built from the original SciFact ***dev*** split, and BEIR *train* comes from original *train*. The original *test* split has no public labels. Section 2.3 checks this mapping from the data instead of assuming it.

| Step | What |
|---|---|
| 2.1 | Download raw files (direct links, with a Hugging Face fallback) |
| 2.2 | Load raw BEIR + original files |
| 2.3 | Alignment checks: corpus, claims, split mapping |
| 2.4 | Canonical corpus table with sentence character spans |
| 2.5 | Queries + qrels tables |
| 2.6 | Evidence mapping (qrel pairs ↔ rationale sentences) |
| 2.7 | Mapping validation (*Fair evaluation protocol, step 1*) |
| 2.8 | Descriptive stats + figures |
| 2.9 | Save processed files + manifest |

### 2.1 Download raw files

In [ ]:
BEIR_URL = "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/scifact.zip"
ORIG_URL = "https://scifact.s3-us-west-2.amazonaws.com/release/latest/data.tar.gz"

BEIR_DIR = RAW_DIR / "beir" / "scifact"
ORIG_DIR = RAW_DIR / "scifact_original" / "data"
BEIR_FILES = [BEIR_DIR / "corpus.jsonl", BEIR_DIR / "queries.jsonl",
              BEIR_DIR / "qrels" / "train.tsv", BEIR_DIR / "qrels" / "test.tsv"]
ORIG_FILES = [ORIG_DIR / f for f in ["corpus.jsonl", "claims_train.jsonl", "claims_dev.jsonl", "claims_test.jsonl"]]
DATA_SOURCE = {}

def _download(url, dest):
    dest.parent.mkdir(parents=True, exist_ok=True)
    print(f"downloading {url}")
    req = urllib.request.Request(url, headers={"User-Agent": "libbot-is424"})
    with urllib.request.urlopen(req, timeout=120) as r, open(dest, "wb") as f:
        shutil.copyfileobj(r, f)
    return dest

def _safe_extract_tar(tar_path, out_dir):
    with tarfile.open(tar_path) as tf:
        for m in tf.getmembers():
            target = (out_dir / m.name).resolve()
            assert str(target).startswith(str(out_dir.resolve())), f"unsafe path in archive: {m.name}"
        tf.extractall(out_dir)

def _beir_from_hf():
    """Fallback: rebuild the BEIR raw layout from the Hugging Face copies."""
    from datasets import load_dataset
    corpus = load_dataset("BeIR/scifact", data_files={"corpus": "corpus/*.parquet"}, split="corpus")
    queries = load_dataset("BeIR/scifact", data_files={"queries": "queries/*.parquet"}, split="queries")
    qrels = load_dataset("BeIR/scifact-qrels")
    write_jsonl([{"_id": str(r["_id"]), "title": r["title"], "text": r["text"]} for r in corpus], BEIR_DIR / "corpus.jsonl")
    write_jsonl([{"_id": str(r["_id"]), "text": r["text"]} for r in queries], BEIR_DIR / "queries.jsonl")
    for s in SPLITS:
        df = qrels[s].to_pandas()
        (BEIR_DIR / "qrels").mkdir(parents=True, exist_ok=True)
        df.columns = ["query-id", "corpus-id", "score"]
        df.to_csv(BEIR_DIR / "qrels" / f"{s}.tsv", sep="\t", index=False)

def _original_from_hf():
    """Fallback: rebuild the original raw layout from Hugging Face (one row per evidence set there)."""
    from datasets import load_dataset
    rev = "refs/convert/parquet"   # allenai/scifact is a script dataset; this revision has plain parquet
    corpus = load_dataset("allenai/scifact", revision=rev, data_files={"train": "corpus/train/*.parquet"}, split="train")
    write_jsonl([{"doc_id": int(r["doc_id"]), "title": r["title"], "abstract": list(r["abstract"]),
                  "structured": bool(r["structured"])} for r in corpus], ORIG_DIR / "corpus.jsonl")
    for hf_split, name in [("train", "train"), ("validation", "dev"), ("test", "test")]:
        claims = {}
        for r in load_dataset("allenai/scifact", revision=rev, data_files={hf_split: f"claims/{hf_split}/*.parquet"},
                              split=hf_split):
            c = claims.setdefault(int(r["id"]), {"id": int(r["id"]), "claim": r["claim"], "evidence": {},
                                                 "cited_doc_ids": list(r.get("cited_doc_ids") or [])})
            if r.get("evidence_doc_id"):
                c["evidence"].setdefault(str(r["evidence_doc_id"]), []).append(
                    {"sentences": list(r["evidence_sentences"]), "label": r["evidence_label"]})
        write_jsonl(list(claims.values()), ORIG_DIR / f"claims_{name}.jsonl")

def fetch_beir():
    if all(p.exists() for p in BEIR_FILES):
        return "cached"
    try:
        z = _download(BEIR_URL, RAW_DIR / "beir" / "scifact.zip")
        with zipfile.ZipFile(z) as zf:
            zf.extractall(RAW_DIR / "beir")
        return BEIR_URL
    except Exception as e:
        print(f"  direct download failed ({e!r}); trying Hugging Face …")
        _beir_from_hf()
        return "huggingface:BeIR/scifact + BeIR/scifact-qrels"

def fetch_original():
    if all(p.exists() for p in ORIG_FILES):
        return "cached"
    try:
        t = _download(ORIG_URL, RAW_DIR / "scifact_original" / "data.tar.gz")
        _safe_extract_tar(t, RAW_DIR / "scifact_original")
        return ORIG_URL
    except Exception as e:
        print(f"  direct download failed ({e!r}); trying Hugging Face …")
        _original_from_hf()
        return "huggingface:allenai/scifact (parquet)"

DATA_SOURCE["beir"] = fetch_beir()
DATA_SOURCE["original"] = fetch_original()
for p in BEIR_FILES + ORIG_FILES:
    check(f"raw file present: {p.relative_to(ROOT)}", p.exists(), hard=True)
DATA_SOURCE

> **If both download routes fail** (e.g. a firewall), download manually and place the files so these paths exist:
> `data/raw/beir/scifact/{corpus.jsonl, queries.jsonl, qrels/train.tsv, qrels/test.tsv}` from the BEIR zip, and
> `data/raw/scifact_original/data/{corpus.jsonl, claims_train.jsonl, claims_dev.jsonl, claims_test.jsonl}` from the SciFact `data.tar.gz`. Then re-run.

### 2.2 Load raw files

In [ ]:
# ---- BEIR ----
beir_corpus = (pd.DataFrame(read_jsonl(BEIR_DIR / "corpus.jsonl"))
               .rename(columns={"_id": "doc_id"})[["doc_id", "title", "text"]])
beir_corpus["doc_id"] = beir_corpus["doc_id"].astype(str)
beir_corpus[["title", "text"]] = beir_corpus[["title", "text"]].fillna("")

beir_queries = (pd.DataFrame(read_jsonl(BEIR_DIR / "queries.jsonl"))
                .rename(columns={"_id": "query_id", "text": "claim"})[["query_id", "claim"]])
beir_queries["query_id"] = beir_queries["query_id"].astype(str)

beir_qrels = []
for s in SPLITS:
    q = pd.read_csv(BEIR_DIR / "qrels" / f"{s}.tsv", sep="\t", dtype=str)
    q.columns = ["query_id", "doc_id", "relevance"]
    q["relevance"] = q["relevance"].astype(int)
    q["split"] = s
    beir_qrels.append(q)
beir_qrels = pd.concat(beir_qrels, ignore_index=True)

# ---- Original SciFact ----
orig_corpus = {str(d["doc_id"]): d for d in read_jsonl(ORIG_DIR / "corpus.jsonl")}
ORIG_SPLITS = ["train", "dev", "test"]
orig_claims = {s: {str(c["id"]): c for c in read_jsonl(ORIG_DIR / f"claims_{s}.jsonl")} for s in ORIG_SPLITS}

raw_counts = pd.Series({
    "BEIR corpus docs": len(beir_corpus),
    "BEIR queries": len(beir_queries),
    "BEIR train qrel rows": int((beir_qrels.split == "train").sum()),
    "BEIR train queries w/ qrels": beir_qrels.loc[beir_qrels.split == "train", "query_id"].nunique(),
    "BEIR test qrel rows": int((beir_qrels.split == "test").sum()),
    "BEIR test queries w/ qrels": beir_qrels.loc[beir_qrels.split == "test", "query_id"].nunique(),
    "Original corpus docs": len(orig_corpus),
    **{f"Original {s} claims": len(orig_claims[s]) for s in ORIG_SPLITS},
    **{f"Original {s} claims with evidence": sum(bool(c.get("evidence")) for c in orig_claims[s].values()) for s in ORIG_SPLITS},
}, name="count")
raw_counts.to_frame()

### 2.3 Alignment checks

Expected sizes (from the SciFact paper and BEIR): corpus 5,183 · BEIR queries 1,109 · BEIR test queries 300 · original claims 809 train / 300 dev / 300 test.
*(The proposal slide's 1,261 / 450 / 300 counts rows in the flattened Hugging Face version, with one row per evidence set, so claims with several evidence documents appear more than once. Counted per claim it is 809 / 300 / 300.)*

Count mismatches are **warnings** (useful if a dataset version changes). Broken joins are **hard failures**.

In [ ]:
EXPECTED = {"corpus": 5183, "beir_queries": 1109, "beir_test_queries": 300,
            "orig_train": 809, "orig_dev": 300, "orig_test": 300}
check("BEIR corpus size", len(beir_corpus) == EXPECTED["corpus"], f"{len(beir_corpus)} vs {EXPECTED['corpus']}")
check("Original corpus size", len(orig_corpus) == EXPECTED["corpus"], f"{len(orig_corpus)} vs {EXPECTED['corpus']}")
check("BEIR queries", len(beir_queries) == EXPECTED["beir_queries"], f"{len(beir_queries)} vs {EXPECTED['beir_queries']}")
n_test_q = beir_qrels.loc[beir_qrels.split == "test", "query_id"].nunique()
check("BEIR test queries with qrels", n_test_q == EXPECTED["beir_test_queries"], f"{n_test_q} vs {EXPECTED['beir_test_queries']}")
for s in ORIG_SPLITS:
    check(f"Original {s} claims", len(orig_claims[s]) == EXPECTED[f"orig_{s}"], f"{len(orig_claims[s])} vs {EXPECTED[f'orig_{s}']}")

# ---- corpus: same documents, same text? ----
beir_ids, orig_ids = set(beir_corpus.doc_id), set(orig_corpus)
check("corpus doc_id sets identical (BEIR vs original)", beir_ids == orig_ids,
      f"only-BEIR={len(beir_ids - orig_ids)}, only-original={len(orig_ids - beir_ids)}", hard=False)
common = beir_corpus[beir_corpus.doc_id.isin(orig_ids)]
title_match = np.mean([norm_ws(t) == norm_ws(orig_corpus[d]["title"]) for d, t in zip(common.doc_id, common.title)])
text_match = np.mean([norm_ws(x) == norm_ws(" ".join(orig_corpus[d]["abstract"])) for d, x in zip(common.doc_id, common.text)])
check("BEIR title == original title (whitespace-normalised)", title_match == 1.0, f"{title_match:.2%} match")
check("BEIR text == ' '.join(original abstract sentences)", text_match == 1.0, f"{text_match:.2%} match")

# ---- claims: BEIR query ids/text match original claims ----
all_orig = {qid: (s, c) for s in ORIG_SPLITS for qid, c in orig_claims[s].items()}
in_orig = beir_queries.query_id.isin(all_orig)
check("every BEIR query_id exists in original claims", in_orig.all(), f"{(~in_orig).sum()} missing", hard=True)
claim_match = np.mean([norm_ws(c) == norm_ws(all_orig[q][1]["claim"]) for q, c in zip(beir_queries.query_id, beir_queries.claim)])
check("BEIR query text == original claim text", claim_match == 1.0, f"{claim_match:.2%} match")

# ---- split mapping, derived from the data ----
overlap = pd.DataFrame({os_: {bs: beir_qrels.loc[beir_qrels.split == bs, "query_id"].drop_duplicates().isin(orig_claims[os_]).sum()
                             for bs in SPLITS} for os_ in ORIG_SPLITS})
overlap.index.name = "BEIR split ↓ / original split →"
display(overlap)
SPLIT_MAP = {bs: overlap.loc[bs].idxmax() for bs in SPLITS}
for bs, os_ in SPLIT_MAP.items():
    n = beir_qrels.loc[beir_qrels.split == bs, "query_id"].nunique()
    check(f"BEIR {bs} queries all come from original {os_}", overlap.loc[bs, os_] == n,
          f"{overlap.loc[bs, os_]}/{n}", hard=True)
check("BEIR test ↔ original dev (expected)", SPLIT_MAP.get("test") == "dev", f"SPLIT_MAP={SPLIT_MAP}")
print("SPLIT_MAP =", SPLIT_MAP)

### 2.4 Canonical corpus table with sentence character spans

`doc_text = (title + " " + text).strip()` is **the** string every method indexes or encodes (it matches the BEIR / Sentence-Transformers convention). Each original abstract sentence is located inside `doc_text`. An exact match is tried first; if that fails, a whitespace-tolerant match is used. This gives `sent_start[i]`, `sent_end[i]` for sentence `i`.

In [ ]:
def build_doc_text(title, text, sep=DOC_SEP):
    return (f"{title}{sep}{text}" if title else text).strip()

def locate_sentences(doc_text, sentences, start=0):
    """Return (starts, ends) char offsets of each sentence inside doc_text, searching left-to-right.
    -1 means the sentence could not be located."""
    starts, ends, cursor = [], [], start
    for sent in sentences:
        s = str(sent).strip()
        if not s:
            starts.append(-1); ends.append(-1); continue
        i = doc_text.find(s, cursor)
        if i >= 0:
            j = i + len(s)
        else:
            m = re.compile(r"\s+".join(map(re.escape, s.split()))).search(doc_text, cursor)
            if not m:
                starts.append(-1); ends.append(-1); continue
            i, j = m.start(), m.end()
        starts.append(i); ends.append(j); cursor = j
    return starts, ends

rows = []
for d in beir_corpus.itertuples(index=False):
    o = orig_corpus.get(d.doc_id, {})
    sents = [str(s) for s in o.get("abstract", [])]
    doc_text = build_doc_text(d.title, d.text)
    body_offset = len(f"{d.title}{DOC_SEP}".lstrip()) if d.title else 0   # some titles have leading spaces that strip() removes
    st, en = locate_sentences(doc_text, sents, start=min(body_offset, len(doc_text)))
    rows.append({"doc_id": d.doc_id, "title": d.title, "text": d.text, "doc_text": doc_text,
                 "sentences": sents, "sent_start": st, "sent_end": en,
                 "n_sentences": len(sents), "n_chars": len(doc_text), "n_words": len(doc_text.split()),
                 "structured": bool(o.get("structured", False))})
corpus = pd.DataFrame(rows)
CORPUS_IDX = {d: i for i, d in enumerate(corpus.doc_id)}

n_unlocated = int(sum(sum(s < 0 for s in st) for st in corpus.sent_start))
n_sent_total = int(corpus.n_sentences.sum())
check("all abstract sentences located in doc_text", n_unlocated == 0, f"{n_unlocated}/{n_sent_total} not located")
check("every doc has sentence segmentation", (corpus.n_sentences > 0).all(), f"{(corpus.n_sentences == 0).sum()} docs without sentences")
corpus[["doc_id", "title", "n_sentences", "n_words", "structured"]].head()

### 2.5 Queries and qrels tables

In [ ]:
qrels = beir_qrels[["query_id", "doc_id", "relevance", "split"]].copy()
check("every qrel doc exists in corpus", qrels.doc_id.isin(CORPUS_IDX).all(),
      f"{(~qrels.doc_id.isin(CORPUS_IDX)).sum()} missing", hard=True)
check("no duplicate qrel rows", not qrels.duplicated(["query_id", "doc_id", "split"]).any(), hard=True)
q_split = qrels.drop_duplicates("query_id").set_index("query_id")["split"]
check("no query appears in both train and test qrels", qrels.groupby("query_id").split.nunique().max() == 1, hard=True)

queries = beir_queries.copy()
queries["split"] = queries.query_id.map(q_split).fillna("unused")
queries["orig_split"] = queries.query_id.map(lambda q: all_orig[q][0])
display(pd.crosstab(queries.split, queries.orig_split, margins=True))
print("'unused' = BEIR queries with no qrels (not evaluated, following the BEIR convention)")

### 2.6 Evidence mapping

For every qrel pair (claim, relevant document), find the original annotation `claims_<orig_split>.jsonl → evidence[doc_id]`. That is a list of **rationale sets**, each a list of sentence indices plus a label. We keep:

- `evidence_sentence_ids`: the union of all rationale sets (sorted), with `ev_char_start/ev_char_end` as the matching spans in `doc_text`
- `label`: SUPPORT / CONTRADICT (`""` if the pair has no rationale)
- one row per evidence sentence in `evidence_sentences` (keeps the rationale-set structure for anyone who needs it)

In [ ]:
pair_rows, sent_rows = [], []
for r in qrels.itertuples(index=False):
    osplit = SPLIT_MAP[r.split]
    claim = orig_claims[osplit].get(r.query_id, {})
    ev_sets = (claim.get("evidence") or {}).get(r.doc_id, [])
    doc = corpus.iloc[CORPUS_IDX[r.doc_id]]
    labels = sorted({e["label"] for e in ev_sets})
    sent_ids = sorted({int(i) for e in ev_sets for i in e["sentences"]})
    in_range = [i for i in sent_ids if 0 <= i < doc.n_sentences]
    starts = [int(doc.sent_start[i]) for i in in_range]
    ends = [int(doc.sent_end[i]) for i in in_range]
    located = [s for s in starts if s >= 0]
    pair_rows.append({
        "query_id": r.query_id, "doc_id": r.doc_id, "split": r.split, "orig_split": osplit,
        "relevance": int(r.relevance), "claim": claim.get("claim", ""),
        "has_evidence": bool(sent_ids),
        "label": labels[0] if len(labels) == 1 else ("|".join(labels) if labels else ""),
        "n_rationale_sets": len(ev_sets),
        "evidence_sentence_ids": sent_ids, "n_evidence_sentences": len(sent_ids),
        "n_doc_sentences": int(doc.n_sentences), "doc_n_chars": int(doc.n_chars),
        "ev_char_start": starts, "ev_char_end": ends,
        "spans_ok": bool(sent_ids) and len(in_range) == len(sent_ids) and len(located) == len(starts),
        "first_evidence_sentence": min(sent_ids) if sent_ids else -1,
        "last_evidence_sentence": max(sent_ids) if sent_ids else -1,
        "first_evidence_char": min(located) if located else -1,
        "last_evidence_char": max(e for s, e in zip(starts, ends) if s >= 0) if located else -1,
        "rel_position_first": (min(sent_ids) / doc.n_sentences) if sent_ids and doc.n_sentences else np.nan,
    })
    for k, e in enumerate(ev_sets):
        for i in e["sentences"]:
            i = int(i)
            ok = 0 <= i < doc.n_sentences
            sent_rows.append({
                "query_id": r.query_id, "doc_id": r.doc_id, "split": r.split, "label": e["label"],
                "rationale_set": k, "sentence_idx": i,
                "char_start": int(doc.sent_start[i]) if ok else -1,
                "char_end": int(doc.sent_end[i]) if ok else -1,
                "sentence_text": doc.sentences[i] if ok else "",
            })

qrel_pairs = pd.DataFrame(pair_rows)
evidence_sentences = pd.DataFrame(sent_rows)

# evidence annotations in the mapped original splits that do NOT appear as BEIR qrels
qrel_keys = set(zip(qrels.query_id, qrels.doc_id))
ev_not_in_qrels = [(qid, did) for bs, os_ in SPLIT_MAP.items() for qid, c in orig_claims[os_].items()
                   for did in (c.get("evidence") or {}) if (qid, did) not in qrel_keys]
qrel_pairs.head(3)

### 2.7 Mapping validation  *(Fair evaluation protocol, step 1: validate the rationale mapping before using it)*

In [ ]:
ev = qrel_pairs[qrel_pairs.has_evidence]
bad_range = sum(any(i >= n for i in ids) for ids, n in zip(ev.evidence_sentence_ids, ev.n_doc_sentences))
check("evidence sentence indices are within the abstract", bad_range == 0, f"{bad_range} pairs out of range", hard=True)
check("all evidence spans located in doc_text", ev.spans_ok.all(), f"{(~ev.spans_ok).sum()} pairs with unlocated spans")

# span text must reproduce the annotated sentence exactly (up to whitespace)
loc = evidence_sentences[evidence_sentences.char_start >= 0]
span_ok = [norm_ws(corpus.doc_text.iat[CORPUS_IDX[d]][s:e]) == norm_ws(t)
           for d, s, e, t in zip(loc.doc_id, loc.char_start, loc.char_end, loc.sentence_text)]
check("doc_text[char_start:char_end] == evidence sentence", all(span_ok), f"{len(span_ok) - sum(span_ok)} mismatches", hard=True)

check("one label per (claim, doc) pair", not ev.label.str.contains(r"\|").any(),
      f"{ev.label.str.contains(r'[|]').sum()} pairs with mixed labels")
check("every evidence annotation (mapped splits) appears in BEIR qrels", len(ev_not_in_qrels) == 0,
      f"{len(ev_not_in_qrels)} annotated pairs not in qrels")
# qrel pairs without a rationale are cited docs of NOINFO claims (BEIR counts cited docs as relevant)
no_ev = qrel_pairs[~qrel_pairs.has_evidence]
cited = [int(d) in orig_claims[o][q].get("cited_doc_ids", []) for q, d, o in zip(no_ev.query_id, no_ev.doc_id, no_ev.orig_split)]
check("every rationale-free qrel pair is a cited doc of its claim", all(cited),
      f"{len(no_ev)} such pairs, {len(cited) - sum(cited)} not cited — kept for retrieval eval, excluded from the evidence diagnostic")

summary = (qrel_pairs.groupby("split")
           .agg(queries=("query_id", "nunique"), qrel_pairs=("doc_id", "size"),
                pairs_with_evidence=("has_evidence", "sum"),
                SUPPORT=("label", lambda s: (s == "SUPPORT").sum()),
                CONTRADICT=("label", lambda s: (s == "CONTRADICT").sum()),
                mean_evidence_sents=("n_evidence_sentences", lambda s: s[s > 0].mean()),
                mean_rationale_sets=("n_rationale_sets", lambda s: s[s > 0].mean()))
           .reindex(SPLITS))
summary["evidence_coverage"] = summary.pairs_with_evidence / summary.qrel_pairs
summary.round(3)

### 2.8 Descriptive statistics

In [ ]:
test_ev = qrel_pairs[(qrel_pairs.split == "test") & qrel_pairs.spans_ok]
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))

axes[0].hist(corpus.n_words, bins=50, color="#5b7553")
axes[0].set(title="Document length (title + abstract)", xlabel="whitespace words", ylabel="documents")
axes[0].axvline(corpus.n_words.median(), color="k", ls="--", lw=1, label=f"median {corpus.n_words.median():.0f}")
axes[0].legend()

axes[1].hist(test_ev.rel_position_first, bins=np.linspace(0, 1, 21), color="#8c6d46")
axes[1].set(title="Where evidence starts (test pairs)", xlabel="first evidence sentence / #sentences", ylabel="pairs")

axes[2].hist(test_ev.first_evidence_char, bins=40, color="#46618c")
axes[2].set(title="First evidence character offset (test pairs)", xlabel="char offset in doc_text", ylabel="pairs")
plt.tight_layout()
fig.savefig(FIG_DIR / "s2_corpus_and_evidence_position.png", dpi=150)
plt.show()

pd.DataFrame({
    "doc words": corpus.n_words.describe(),
    "doc sentences": corpus.n_sentences.describe(),
    "evidence sents / pair": qrel_pairs.loc[qrel_pairs.has_evidence, "n_evidence_sentences"].describe(),
    "first evidence sentence idx": qrel_pairs.loc[qrel_pairs.has_evidence, "first_evidence_sentence"].describe(),
}).round(2)

### 2.9 Save processed files + manifest

In [ ]:
outputs = {"corpus": corpus, "queries": queries, "qrels": qrels,
           "qrel_pairs": qrel_pairs, "evidence_sentences": evidence_sentences}
for name, df in outputs.items():
    df.to_parquet(PROCESSED_FILES[name], index=False)

manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "data_source": DATA_SOURCE, "split_map_beir_to_original": SPLIT_MAP, "doc_sep": DOC_SEP,
    "counts": {"docs": len(corpus), "queries": len(queries),
               **{f"{s}_eval_queries": int((queries.split == s).sum()) for s in SPLITS},
               **{f"{s}_qrel_pairs": int((qrels.split == s).sum()) for s in SPLITS},
               **{f"{s}_pairs_with_evidence": int(qrel_pairs[(qrel_pairs.split == s)].has_evidence.sum()) for s in SPLITS},
               "evidence_sentence_rows": len(evidence_sentences),
               "annotated_pairs_not_in_qrels": len(ev_not_in_qrels)},
    "files": {n: {"path": str(PROCESSED_FILES[n].relative_to(ROOT)), "sha256": sha256_file(PROCESSED_FILES[n])} for n in outputs},
    "raw_sha256": {str(p.relative_to(ROOT)): sha256_file(p) for p in BEIR_FILES + ORIG_FILES},
    "checks": CHECK_LOG, "environment": ENV,
}
(PROC_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2))
_CORPUS_IDS = None   # refresh cached corpus ids used by validate_run()
print(json.dumps(manifest["counts"], indent=2))
n_warn = sum(c["status"] != "PASS" for c in CHECK_LOG)
print(f"\nSection 2 done — {len(CHECK_LOG)} checks, {n_warn} warnings. Outputs in {PROC_DIR.relative_to(ROOT)}/")

---
# Checkpoint A · Load shared data  *(start here if you are skipping Section 2)*

Run Section 1, then this cell. It loads everything Sections 3–10 need from `data/processed/`.

| Variable | What |
|---|---|
| `corpus` | DataFrame, one row per doc. **Keep this order**: score matrices use columns in `DOC_IDS` order |
| `DOC_IDS`, `DOC_TEXTS` | lists aligned with `corpus` (the canonical `doc_text`) |
| `Q[split]` | evaluation queries for `split` (`query_id`, `claim`) |
| `QRELS[split]` | `{query_id: {doc_id: relevance}}` |
| `qrel_pairs`, `evidence_sentences` | evidence mapping (diagnostic use; test evidence never used for tuning) |

In [ ]:
corpus, queries, qrels, qrel_pairs, evidence_sentences = load_processed(
    "corpus", "queries", "qrels", "qrel_pairs", "evidence_sentences")
DOC_IDS = corpus["doc_id"].tolist()
DOC_TEXTS = corpus["doc_text"].tolist()
CORPUS_IDX = {d: i for i, d in enumerate(DOC_IDS)}
Q = {s: eval_queries(s, queries) for s in SPLITS}
QRELS = {s: qrels_dict(s, qrels) for s in SPLITS}
print(f"{len(DOC_IDS)} docs | " + " | ".join(f"{s}: {len(Q[s])} queries, {sum(map(len, QRELS[s].values()))} qrels" for s in SPLITS))
run_status()

---
# 3 · BM25 baseline  *(Member 3)*

Lexical reference method: **BM25Okapi** (`rank-bm25`) over the full tokenised `doc_text` (no truncation).

**What BM25 does.** For a query $q$ and document $d$,

$$\text{BM25}(q,d) = \sum_{t \in q} \text{IDF}(t)\cdot\frac{f(t,d)\,(k_1+1)}{f(t,d) + k_1\left(1 - b + b\,\frac{|d|}{\text{avgdl}}\right)}$$

- $f(t,d)$: how often term $t$ occurs in $d$. $k_1$ caps how much repeated terms help (term-frequency saturation).
- $b$ controls length normalisation: long abstracts are penalised so they don't win just by containing more words.
- $\text{IDF}(t) = \log\frac{N - n_t + 0.5}{n_t + 0.5}$ rewards rare terms. In `rank-bm25`, a term that occurs in more than half the documents would get a negative IDF. Its IDF is floored at `epsilon × mean IDF` instead.

**Why it matters for the research question.** BM25 has no input-length limit: it sees every word of every abstract. That makes it the full-visibility reference next to prefix MiniLM, which only sees the first 254 word-pieces. Its rankings also feed both hybrids (Section 7).

**Inputs (from Checkpoint A / disk)**
- `DOC_IDS`, `DOC_TEXTS` (index these, in this order)
- `Q["train"]`, `Q["test"]` (`query_id`, `claim`)
- `QRELS[split]` for a quick self-check (train is also used to pick the tokeniser; test is **not**)

**Required outputs**
- `save_run(run, "bm25", "train")` and `save_run(run, "bm25", "test")` (build `run` with `scores_to_run(score_matrix, Q[split].query_id, DOC_IDS)`; keep **raw** BM25 scores)
- `log_efficiency("bm25", indexing_time_s=..., mean_latency_ms=..., p95_latency_ms=..., n_queries=...)`
- Extra: `results/metrics/bm25_tokeniser_sweep__train.csv` (evidence for the tokeniser choice)

**Notes & checklist**
- [✅] Choose and document the tokeniser and keep it identical for docs and queries → lowercase, `[a-z0-9]+`, sklearn English stop-words, Porter stemming (chosen on **train**, see 3.1b)
- [✅] Record BM25 parameters → rank-bm25 defaults `k1=1.5, b=0.75, epsilon=0.25` (not tuned)
- [✅] Warm up before timing the latency; time per query, then report mean and p95 (3.3)
- [✅] Sanity check against the proposal's preliminary test result: Recall@10 ≈ 0.776, nDCG@10 ≈ 0.652 (3.2; outcome in 3.4)

### 3.0 Look at the inputs

A read-only look at what BM25 will index and score. Checks to make:

- **Corpus:** one row per abstract. `doc_text` = title + abstract, the exact string we tokenise. The order of `corpus` defines the columns of every score matrix.
- **Queries:** short scientific claims. They are often one sentence, so a single matching rare term (a gene or drug name) can dominate the BM25 score.
- **Qrels:** most claims have exactly one relevant abstract, so Recall@10 is close to "was the one right document in the top 10?".

In [ ]:
# ===================== 3.0 Inspect the inputs (read-only) =====================
view_cols = ["doc_id", "title", "doc_text", "n_words"]
print(f"corpus: {len(corpus):,} docs × {corpus.shape[1]} columns")
print("first 5 docs:"); display(corpus[view_cols].head(5))
print("last 5 docs:");  display(corpus[view_cols].tail(5))
print("doc length in words (doc_text):")
display(corpus["n_words"].describe(percentiles=[0.5, 0.9, 0.99]).round(1).to_frame().T)

In [ ]:
for split in SPLITS:
    print(f"--- {split}: {len(Q[split])} evaluation queries (head 3 / tail 3)")
    display(pd.concat([Q[split][["query_id", "claim"]].head(3), Q[split][["query_id", "claim"]].tail(3)]))

claim_len = pd.DataFrame({s: Q[s]["claim"].str.split().str.len().describe().round(1) for s in SPLITS}).T
print("claim length in words:"); display(claim_len)

rel_per_q = pd.DataFrame({s: pd.Series([len(r) for r in QRELS[s].values()]).value_counts().sort_index()
                          for s in SPLITS}).fillna(0).astype(int)
rel_per_q.index.name = "relevant docs per query"
print("queries by number of relevant docs:"); display(rel_per_q)

### 3.1 Tokeniser and BM25 index

BM25 matches **tokens**, so the tokeniser decides what counts as a match. The same function is applied to documents and queries.

| Step | Setting | Why |
|---|---|---|
| Lowercase | yes | "Vitamin D" and "vitamin d" should match |
| Split | regex `[a-z0-9]+` | splits on punctuation and hyphens, e.g. `IL-6` → `il`, `6` and `p<0.05` → `p`, `0`, `05`, so `IL-6` in a claim matches `IL-6,` in an abstract |
| Stop-words | scikit-learn `ENGLISH_STOP_WORDS` (318 words) | removes "the", "of", "in"; these carry little signal and add noise to long abstracts |
| Stemming | NLTK Porter | `mutations` / `mutated` / `mutation` → `mutat`, so claims and abstracts match across word forms |

**Pilot provenance.** The proposal's pilot (Recall@10 0.776 / nDCG@10 0.652, 26 Sep, Apple M2 Pro) was meant to have its settings in `results/bm25_pilot_provenance.json`. That file is **not in the repo**, so the pilot's tokeniser is unknown. Following the fair protocol, the tokeniser is instead chosen from a small set of candidates **using train queries only** (3.1b). Test is scored once, after the choice is frozen (3.2).

**BM25 parameters.** `k1=1.5, b=0.75, epsilon=0.25` (rank-bm25 defaults), kept fixed and not tuned, as the checklist asks.

In [ ]:
# ===================== 3.1a Tokeniser factory (one function for docs AND queries) =====================
import pickle
from functools import lru_cache
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from nltk.stem import PorterStemmer   # PorterStemmer needs no nltk data download

BM25_PARAMS = dict(k1=1.5, b=0.75, epsilon=0.25)   # rank-bm25 defaults, recorded explicitly, not tuned

_PORTER = PorterStemmer()
@lru_cache(maxsize=None)
def _porter(w):
    return _PORTER.stem(w)

def make_tokeniser(lowercase=True, pattern=r"[a-z0-9]+", stopwords=None, stemmer=None):
    tok_re = re.compile(pattern)
    stop = frozenset(ENGLISH_STOP_WORDS) if stopwords == "sklearn_english" else frozenset()
    stem = _porter if stemmer == "porter" else (lambda w: w)
    def tokenise(text):
        text = text.lower() if lowercase else text
        return [stem(w) for w in tok_re.findall(text) if w not in stop]
    return tokenise

TOKENISER_CANDIDATES = {
    "whitespace":      dict(lowercase=True, pattern=r"\S+",       stopwords=None,              stemmer=None),
    "alnum":           dict(lowercase=True, pattern=r"[a-z0-9]+", stopwords=None,              stemmer=None),
    "alnum+stop":      dict(lowercase=True, pattern=r"[a-z0-9]+", stopwords="sklearn_english", stemmer=None),
    "alnum+stem":      dict(lowercase=True, pattern=r"[a-z0-9]+", stopwords=None,              stemmer="porter"),
    "alnum+stop+stem": dict(lowercase=True, pattern=r"[a-z0-9]+", stopwords="sklearn_english", stemmer="porter"),
}

# What each candidate does to a real (train) claim
example_claim = Q["train"]["claim"].iloc[0]
print("claim:", example_claim)
for name, cfg in TOKENISER_CANDIDATES.items():
    print(f"  {name:<16} {make_tokeniser(**cfg)(example_claim)}")

#### 3.1b Tokeniser selection on **train** only

Each candidate gets its own index, scores the 809 train claims, and is compared with the temporary `quick_metrics` self-check. The winner by train nDCG@10 is frozen below. Test qrels are **not** used here.

*Our run (rank-bm25 defaults), train nDCG@10:* whitespace 0.590 → alnum 0.658 → +stop 0.663 → +stem 0.690 → **+stop+stem 0.694**. Splitting on punctuation is the biggest gain, and stemming adds about 0.03. This matches the drafted default.

In [ ]:
# ===================== 3.1b Tokeniser sweep on TRAIN (selection only; takes ~1–2 min) =====================
def quick_metrics(run, qrels, k=K):
    # Temporary self-check only. Member 1's Section 8 is the official evaluator.
    top = run[run["rank"] <= k].groupby("query_id")["doc_id"].apply(list).to_dict()
    rec, ndcg = [], []
    for qid, rel in qrels.items():
        ranked = top.get(qid, [])
        hits = [1.0 if d in rel else 0.0 for d in ranked]
        rec.append(sum(hits) / len(rel))
        dcg = sum(h / np.log2(i + 2) for i, h in enumerate(hits))
        idcg = sum(1 / np.log2(i + 2) for i in range(min(len(rel), k)))
        ndcg.append(dcg / idcg)
    return {f"recall@{k}": float(np.mean(rec)), f"ndcg@{k}": float(np.mean(ndcg)), "n_queries": len(rec)}

sweep_rows = []
for name, cfg in TOKENISER_CANDIDATES.items():
    tok = make_tokeniser(**cfg)
    idx = BM25Okapi([tok(t) for t in DOC_TEXTS], **BM25_PARAMS)
    S_tr = np.vstack([idx.get_scores(tok(c)) for c in Q["train"]["claim"]])
    m = quick_metrics(scores_to_run(S_tr, Q["train"]["query_id"].tolist(), DOC_IDS, depth=K), QRELS["train"])
    sweep_rows.append({"tokeniser": name, **m, "vocab": len(idx.idf), "avg_doc_tokens": round(idx.avgdl, 1)})
    print(f"{name:<16} train R@10={m['recall@10']:.4f}  nDCG@10={m['ndcg@10']:.4f}")
del idx, S_tr

tok_sweep = pd.DataFrame(sweep_rows).sort_values("ndcg@10", ascending=False).reset_index(drop=True)
tok_sweep.to_csv(METRICS_DIR / "bm25_tokeniser_sweep__train.csv", index=False)
display(tok_sweep)

# ---- FREEZE the choice (hard-coded so later runs don't silently change it) ----
BM25_TOKENISER_NAME = "alnum+stop+stem"
BM25_TOKENISER = TOKENISER_CANDIDATES[BM25_TOKENISER_NAME]
check("frozen tokeniser is the best on train nDCG@10", tok_sweep.loc[0, "tokeniser"] == BM25_TOKENISER_NAME,
      f"best on train: {tok_sweep.loc[0, 'tokeniser']}")
print("FROZEN tokeniser:", BM25_TOKENISER_NAME, BM25_TOKENISER, "| params:", BM25_PARAMS)

In [ ]:
# ===================== 3.1c Build the frozen BM25 index (time the indexing) =====================
bm25_tokenise = make_tokeniser(**BM25_TOKENISER)
_porter.cache_clear()        # the sweep warmed the stem cache; clear it so indexing time is a cold build

t0 = time.perf_counter()
doc_tokens = [bm25_tokenise(t) for t in DOC_TEXTS]          # same order as DOC_IDS
bm25 = BM25Okapi(doc_tokens, **BM25_PARAMS)
BM25_INDEX_TIME_S = time.perf_counter() - t0                # = tokenise all docs + build term stats/IDF

bm25_path = CACHE_DIR / "bm25_index.pkl"                    # git-ignored; storage size for the efficiency table
with open(bm25_path, "wb") as f:
    pickle.dump(bm25, f)
BM25_STORAGE_BYTES = bm25_path.stat().st_size

check("BM25 index covers every doc", bm25.corpus_size == len(DOC_IDS), f"{bm25.corpus_size} vs {len(DOC_IDS)}", hard=True)
n_empty = sum(len(t) == 0 for t in doc_tokens)
check("no empty tokenised documents", n_empty == 0, f"{n_empty} empty docs")
print(f"indexed {bm25.corpus_size} docs in {BM25_INDEX_TIME_S:.2f}s | vocab {len(bm25.idf):,} | "
      f"avg doc len {bm25.avgdl:.1f} tokens | pickle {BM25_STORAGE_BYTES/1e6:.1f} MB")

# ---- What the index sees ----
tok_view = pd.DataFrame({"doc_id": DOC_IDS, "n_tokens": [len(t) for t in doc_tokens],
                         "first_15_tokens": [" ".join(t[:15]) for t in doc_tokens]})
print("tokenised docs (head 3 / tail 3):"); display(pd.concat([tok_view.head(3), tok_view.tail(3)]))
print("tokens per doc:"); display(tok_view["n_tokens"].describe(percentiles=[0.5, 0.9, 0.99]).round(1).to_frame().T)

idf = pd.Series(bm25.idf).sort_values()
n_docs_with = Counter(w for t in doc_tokens for w in set(t))
print("10 most common terms (lowest IDF) and 10 rarest terms (highest IDF):")
display(pd.DataFrame({"common_term": idf.index[:10], "common_idf": idf.values[:10].round(3),
                      "common_n_docs": [n_docs_with[w] for w in idf.index[:10]],
                      "rare_term": idf.index[-10:], "rare_idf": idf.values[-10:].round(3),
                      "rare_n_docs": [n_docs_with[w] for w in idf.index[-10:]]}))

### 3.2 Score train + test, save runs

- `bm25.get_scores(tokens)` returns one **raw** BM25 score for every document, giving a `(n_queries × 5,183)` matrix in `DOC_IDS` column order.
- `scores_to_run` keeps the top `RUN_DEPTH = 1000` per query, and `save_run` validates the shared format (string IDs, ranks 1…n, non-increasing scores, the right query set).
- **Scores are not normalised.** Member 6's fusion min-max normalises per query later.
- **Zeros can fill the bottom.** A document that shares no (stemmed, non-stop-word) term with the claim scores exactly 0, and ties are ordered by `doc_id`. With this tokeniser a typical claim still matches about 2,500 abstracts (numbers and common stems such as `cell` match widely), so only about 3% of saved rows are 0. Those come from claims with rare vocabulary that match fewer than 1,000 documents. This is expected.
- Test is scored once, with the tokeniser already frozen. The pilot comparison below is a sanity check, **not** a tuning signal.

In [ ]:
# ===================== 3.2 Score all queries -> score matrix (n_queries x n_docs) -> scores_to_run -> save_run =====================
def bm25_score_matrix(claims):
    return np.vstack([bm25.get_scores(bm25_tokenise(c)) for c in claims])   # (n_queries, n_docs), raw scores

BM25_RESULTS = {}
for split in SPLITS:
    qdf = Q[split]
    S = bm25_score_matrix(qdf["claim"].tolist())
    n_zero_q = int((S.max(axis=1) <= 0).sum())
    check(f"bm25/{split}: every query matches at least one doc", n_zero_q == 0, f"{n_zero_q} queries with all-zero scores")
    run = scores_to_run(S, qdf["query_id"].tolist(), DOC_IDS)
    save_run(run, "bm25", split)
    BM25_RESULTS[split] = quick_metrics(run, QRELS[split])
    n_matched = (S > 0).sum(axis=1)
    print(f"  {split}: {BM25_RESULTS[split]}")
    print(f"  docs with score > 0 per query: median {int(np.median(n_matched))}, min {n_matched.min()}, max {n_matched.max()} "
          f"| zero-score rows in the saved top-{RUN_DEPTH}: {(run.score == 0).mean():.1%}")

# Sanity check against the proposal's preliminary result (test split). Small gaps usually mean a different tokeniser.
r = BM25_RESULTS["test"]
check("test Recall@10 close to proposal pilot (0.776)", abs(r["recall@10"] - 0.776) < 0.01, f"{r['recall@10']:.4f}")
check("test nDCG@10 close to proposal pilot (0.652)", abs(r["ndcg@10"] - 0.652) < 0.01, f"{r['ndcg@10']:.4f}")

In [ ]:
# ---- Look at a saved run: one query's head (top 10) and tail (zero scores) ----
run = load_run("bm25", "test")
print(f"bm25__test.parquet: {len(run):,} rows | {run.query_id.nunique()} queries | dtypes {dict(run.dtypes.astype(str))}")
qid0 = Q["test"]["query_id"].iloc[0]
claim0 = Q["test"].set_index("query_id").loc[qid0, "claim"]
rel0 = QRELS["test"][qid0]
one = (run[run.query_id == qid0]
       .merge(corpus[["doc_id", "title"]], on="doc_id", how="left")
       .assign(relevant=lambda d: d.doc_id.isin(rel0)))
print(f"query {qid0}: {claim0}\nrelevant doc(s): {list(rel0)}")
print("head (top 10):"); display(one.head(10))
print("tail (last 5 of the top 1000):"); display(one.tail(5))

# Where does the first relevant doc land? (counts over test queries)
rel_rows = run.merge(qrels[qrels.split == "test"][["query_id", "doc_id"]], on=["query_id", "doc_id"])
first_rel = rel_rows.groupby("query_id")["rank"].min().reindex(list(QRELS["test"]))
bucket = pd.cut(first_rel, [0, 1, 3, 10, 100, RUN_DEPTH], labels=["1", "2–3", "4–10", "11–100", f"101–{RUN_DEPTH}"])
rank_dist = bucket.value_counts(sort=False).astype(int)
rank_dist[f"not in top {RUN_DEPTH}"] = int(first_rel.isna().sum())
print("rank of the first relevant doc (test queries):"); display(rank_dist.to_frame("queries"))

### 3.3 Warm query latency + efficiency log

**Timing boundary** (Members 4 and 5 should match it so the BM25-vs-MiniLM comparison is fair). The timer runs for **one query at a time**, after a 20-query warm-up, and covers:

1. tokenising the claim (MiniLM equivalent: encoding the query with the model),
2. scoring **every** document (MiniLM: cosine against every cached doc or window embedding, plus MaxP for windows),
3. selecting the top 10.

It **excludes** index building, model loading and file I/O. All 300 test claims are timed 3 times (900 timings), and the mean and p95 are reported. Only the claim text is used, not any qrels.

> **These are BM25-only self-check timings.** The official latency for all five methods is re-measured together in **7.4** (300-claim warm-up, 300 claims × 3) and reported in `results/metrics/final_table.csv`.

`log_efficiency` appends to a shared file. This cell first removes any earlier `bm25` record, so the file always holds **exactly one** BM25 line, and leaves other members' lines untouched.

In [ ]:
# ===================== 3.3 Warm query latency + log_efficiency('bm25', ...) =====================
def bm25_search(claim, k=K):
    s = bm25.get_scores(bm25_tokenise(claim))
    top = np.argpartition(-s, k - 1)[:k]
    return top[np.argsort(-s[top])]

lat_claims = Q["test"]["claim"].tolist()
for c in lat_claims[:20]:          # warm-up, not timed
    bm25_search(c)

N_REPEATS = 3
lat_ms = []
for _ in range(N_REPEATS):
    for c in lat_claims:
        t0 = time.perf_counter()
        bm25_search(c)
        lat_ms.append((time.perf_counter() - t0) * 1000)
lat_ms = np.array(lat_ms)

# keep exactly one bm25 record in the shared log (rerun-safe; other methods' lines untouched)
eff_log = EFF_DIR / "efficiency_log.jsonl"
if eff_log.exists():
    write_jsonl([r for r in read_jsonl(eff_log) if r.get("method") != "bm25"], eff_log)

eff = log_efficiency(
    "bm25",
    indexing_time_s=round(BM25_INDEX_TIME_S, 4),
    mean_latency_ms=round(float(lat_ms.mean()), 4),
    p95_latency_ms=round(float(np.percentile(lat_ms, 95)), 4),
    n_queries=len(lat_claims), n_repeats=N_REPEATS,
    n_embeddings=0, storage_bytes=BM25_STORAGE_BYTES,
    timing_boundary="tokenise query + score all docs + top-10 select; single query; warm",
    tokeniser=BM25_TOKENISER_NAME, tokeniser_config=BM25_TOKENISER, params=BM25_PARAMS,
)
print(json.dumps(eff, indent=2, default=str))
print(f"latency ms: min {lat_ms.min():.2f} | median {np.median(lat_ms):.2f} | mean {lat_ms.mean():.2f} | "
      f"p95 {np.percentile(lat_ms, 95):.2f} | max {lat_ms.max():.2f}")

### 3.4 Section 3 summary (record for the report's method section)

| Item | Value |
|---|---|
| Library | `rank-bm25==0.2.2`, `BM25Okapi` |
| Parameters | `k1 = 1.5`, `b = 0.75`, `epsilon = 0.25` (defaults, not tuned) |
| Tokeniser (`alnum+stop+stem`) | lowercase → regex `[a-z0-9]+` → drop scikit-learn English stop-words → NLTK Porter stem; same for docs and claims |
| How chosen | best train nDCG@10 of 5 candidates (`results/metrics/bm25_tokeniser_sweep__train.csv`); test not used |
| Indexed text | full `doc_text` (title + abstract), no truncation |
| Outputs | `results/runs/bm25__train.parquet` (809 q), `results/runs/bm25__test.parquet` (300 q), 1 `bm25` line in `efficiency_log.jsonl` |
| Efficiency | index pickle 3.5 MB · official indexing time and latency come from 7.4 (`results/metrics/final_table.csv`, hardware in `efficiency_log.jsonl`) |
| Self-check (not official) | train R@10 0.815 / nDCG@10 0.694 · test R@10 **0.820** / nDCG@10 **0.684** |

**Pilot comparison: WARN.** On test, our run gives Recall@10 0.820 and nDCG@10 0.684, against the pilot's 0.776 and 0.652. That is about +0.044 recall and +0.032 nDCG, so both 0.01-tolerance checks in 3.2 show WARN. Our run is *higher*, by about the 0.01–0.05 a tokeniser change typically causes. The train sweep supports this: a whitespace split with no stemming loses about 0.10 train nDCG. The pilot's provenance file (`results/bm25_pilot_provenance.json`) is missing, so its tokeniser can't be reproduced exactly. We keep the train-selected tokeniser and do **not** switch to whichever variant lands closest to the pilot on test. The report should describe the proposal's figures as a preliminary pilot, superseded by Section 8.

**Of the 300 test claims** (3.2 view), the first relevant abstract is ranked 1st for 164, 2nd–10th for 88, 11th–1000th for 41, and missing from the top 1000 for 7.

*These are Member 3's quick self-check numbers. Member 1's Section 8 is the official evaluator.* Latency and indexing time depend on hardware: see the printed 3.3 output or `efficiency_log.jsonl`.

---
# 4 · Prefix MiniLM  *(Member 4)*

Semantic baseline: one embedding per document from `all-MiniLM-L6-v2`. The model sees only the first `MAX_SEQ_LEN` (256) word-pieces of `doc_text`, including [CLS]/[SEP]; everything after that is cut off.

**Inputs (from Checkpoint A / disk)**
- `DOC_IDS`, `DOC_TEXTS` (encode exactly these strings; do **not** re-join title/text yourself)
- `Q[split]` claims, `MODEL_NAME`, `MAX_SEQ_LEN`

**Required outputs**
- Cached doc embeddings: `cache/prefix_doc_emb.npy` (float32, shape 5183×384, rows in `DOC_IDS` order) + `cache/prefix_doc_ids.json`
- Cached query embeddings per split: `cache/query_emb__{split}.npy` (+ ids). Member 5 reuses these, so the query encoding is shared
- `save_run(run, "prefix_minilm", split)` for train and test (cosine scores)
- `log_efficiency("prefix_minilm", indexing_time_s=..., n_embeddings=..., storage_bytes=..., mean_latency_ms=..., p95_latency_ms=...)`

**Notes & checklist**
- [✅] `model.max_seq_length = MAX_SEQ_LEN` (assert it), frozen model, `normalize_embeddings=True` so dot product = cosine
- [✅] Load from the cache if the file exists; otherwise encode and save
- [✅] Latency = query encoding + cosine scoring over the cached matrix, measured after a warm-up
- [✅] Record device (CPU/GPU/MPS) and batch size

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

PREFIX_BATCH_SIZE = 64

if torch.cuda.is_available():
    PREFIX_DEVICE = "cuda"
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    PREFIX_DEVICE = "mps"
else:
    PREFIX_DEVICE = "cpu"

print(f"Loading {MODEL_NAME} on {PREFIX_DEVICE} ...")

prefix_model = SentenceTransformer(
    MODEL_NAME,
    device=PREFIX_DEVICE
)

# MiniLM input limit used in our proposal
prefix_model.max_seq_length = MAX_SEQ_LEN
assert prefix_model.max_seq_length == MAX_SEQ_LEN

# Freeze MiniLM — no fine-tuning
prefix_model.eval()
for p in prefix_model.parameters():
    p.requires_grad_(False)

PREFIX_EMB_DIM = prefix_model.get_sentence_embedding_dimension()

assert PREFIX_EMB_DIM == 384, (
    f"Expected 384-D embeddings, got {PREFIX_EMB_DIM}"
)

print(
    f"Model: {MODEL_NAME}\n"
    f"Device: {PREFIX_DEVICE}\n"
    f"Max sequence length: {MAX_SEQ_LEN}\n"
    f"Embedding dimension: {PREFIX_EMB_DIM}"
)

PREFIX_DOC_EMB_PATH = CACHE_DIR / "prefix_doc_emb.npy"
PREFIX_DOC_IDS_PATH = CACHE_DIR / "prefix_doc_ids.json"
PREFIX_META_PATH = CACHE_DIR / "prefix_doc_emb.meta.json"


def _json_load(path):
    return json.loads(
        Path(path).read_text(encoding="utf-8")
    )


def _write_json(obj, path):
    Path(path).write_text(
        json.dumps(obj, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )


expected_meta = {
    "model_name": MODEL_NAME,
    "max_seq_length": MAX_SEQ_LEN,
    "normalize_embeddings": True,
    "embedding_dim": PREFIX_EMB_DIM,
    "n_documents": len(DOC_IDS),
}


def _prefix_doc_cache_is_valid():

    required_files = [
        PREFIX_DOC_EMB_PATH,
        PREFIX_DOC_IDS_PATH,
        PREFIX_META_PATH
    ]

    if not all(path.exists() for path in required_files):
        return False

    try:
        cached_ids = [
            str(x)
            for x in _json_load(PREFIX_DOC_IDS_PATH)
        ]

        meta = _json_load(PREFIX_META_PATH)

        if cached_ids != [str(x) for x in DOC_IDS]:
            return False

        for key, value in expected_meta.items():
            if meta.get(key) != value:
                return False

        if "indexing_time_s" not in meta:
            return False

        arr = np.load(
            PREFIX_DOC_EMB_PATH,
            mmap_mode="r"
        )

        return (
            arr.shape == (len(DOC_IDS), PREFIX_EMB_DIM)
            and arr.dtype == np.float32
        )

    except Exception:
        return False


if _prefix_doc_cache_is_valid():

    prefix_doc_emb = np.load(
        PREFIX_DOC_EMB_PATH
    )

    prefix_meta = _json_load(
        PREFIX_META_PATH
    )

    prefix_indexing_time_s = float(
        prefix_meta["indexing_time_s"]
    )

    print(
        "Loaded cached Prefix MiniLM "
        "document embeddings."
    )

else:

    print(
        f"Encoding {len(DOC_TEXTS):,} documents..."
    )

    start = time.perf_counter()

    prefix_doc_emb = prefix_model.encode(
        DOC_TEXTS,
        batch_size=PREFIX_BATCH_SIZE,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    prefix_indexing_time_s = (
        time.perf_counter() - start
    )

    prefix_doc_emb = np.asarray(
        prefix_doc_emb,
        dtype=np.float32
    )

    np.save(
        PREFIX_DOC_EMB_PATH,
        prefix_doc_emb
    )

    _write_json(
        [str(x) for x in DOC_IDS],
        PREFIX_DOC_IDS_PATH
    )

    prefix_meta = {
        **expected_meta,
        "indexing_time_s": prefix_indexing_time_s,
        "batch_size": PREFIX_BATCH_SIZE,
        "device": PREFIX_DEVICE
    }

    _write_json(
        prefix_meta,
        PREFIX_META_PATH
    )

    print(
        "Saved Prefix MiniLM "
        "document embeddings."
    )


assert prefix_doc_emb.shape == (
    len(DOC_IDS),
    384
)

assert prefix_doc_emb.dtype == np.float32

doc_norms = np.linalg.norm(
    prefix_doc_emb,
    axis=1
)

assert np.allclose(
    doc_norms,
    1.0,
    atol=1e-4
)

print(
    "Document embedding shape:",
    prefix_doc_emb.shape
)

print(
    f"Indexing time: "
    f"{prefix_indexing_time_s:.3f} s"
)

print(
    f"Embedding cache size: "
    f"{PREFIX_DOC_EMB_PATH.stat().st_size / (1024**2):.2f} MiB"
)

In [ ]:
PREFIX_QUERY_EMB = {}
PREFIX_QUERY_IDS = {}
PREFIX_QUERY_TEXTS = {}

for split in SPLITS:

    qdf = Q[split].reset_index(drop=True)

    query_ids = (
        qdf["query_id"]
        .astype(str)
        .tolist()
    )

    query_texts = (
        qdf["claim"]
        .astype(str)
        .tolist()
    )

    emb_path = (
        CACHE_DIR /
        f"query_emb__{split}.npy"
    )

    ids_path = (
        CACHE_DIR /
        f"query_ids__{split}.json"
    )

    meta_path = (
        CACHE_DIR /
        f"query_emb__{split}.meta.json"
    )

    expected_q_meta = {
        "model_name": MODEL_NAME,
        "max_seq_length": MAX_SEQ_LEN,
        "normalize_embeddings": True,
        "embedding_dim": PREFIX_EMB_DIM,
        "n_queries": len(query_ids),
        "split": split
    }

    cache_ok = False

    if (
        emb_path.exists()
        and ids_path.exists()
        and meta_path.exists()
    ):

        try:

            cached_ids = [
                str(x)
                for x in _json_load(ids_path)
            ]

            meta = _json_load(meta_path)

            arr = np.load(
                emb_path,
                mmap_mode="r"
            )

            cache_ok = (
                cached_ids == query_ids
                and all(
                    meta.get(k) == v
                    for k, v
                    in expected_q_meta.items()
                )
                and arr.shape
                == (
                    len(query_ids),
                    PREFIX_EMB_DIM
                )
                and arr.dtype
                == np.float32
            )

        except Exception:
            cache_ok = False

    if cache_ok:

        q_emb = np.load(
            emb_path
        )

        print(
            f"[{split}] loaded cached "
            "query embeddings"
        )

    else:

        print(
            f"[{split}] encoding "
            f"{len(query_texts):,} queries..."
        )

        start = time.perf_counter()

        q_emb = prefix_model.encode(
            query_texts,
            batch_size=PREFIX_BATCH_SIZE,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True
        )

        query_encode_time_s = (
            time.perf_counter()
            - start
        )

        q_emb = np.asarray(
            q_emb,
            dtype=np.float32
        )

        np.save(
            emb_path,
            q_emb
        )

        _write_json(
            query_ids,
            ids_path
        )

        _write_json(
            {
                **expected_q_meta,
                "encoding_time_s":
                    query_encode_time_s,
                "batch_size":
                    PREFIX_BATCH_SIZE,
                "device":
                    PREFIX_DEVICE
            },
            meta_path
        )

        print(
            f"[{split}] saved query embeddings"
        )

    assert q_emb.shape == (
        len(query_ids),
        384
    )

    assert q_emb.dtype == np.float32

    query_norms = np.linalg.norm(
        q_emb,
        axis=1
    )

    assert np.allclose(
        query_norms,
        1.0,
        atol=1e-4
    )

    PREFIX_QUERY_EMB[split] = q_emb
    PREFIX_QUERY_IDS[split] = query_ids
    PREFIX_QUERY_TEXTS[split] = query_texts


print("\nQuery embedding caches ready:")

for split in SPLITS:
    print(
        split,
        PREFIX_QUERY_EMB[split].shape
    )

In [ ]:
PREFIX_RUNS = {}

for split in SPLITS:

    q_emb = PREFIX_QUERY_EMB[split]

    # Embeddings were L2-normalized,
    # so dot product = cosine similarity
    prefix_scores = (
        q_emb
        @ prefix_doc_emb.T
    )

    assert prefix_scores.shape == (
        len(PREFIX_QUERY_IDS[split]),
        len(DOC_IDS)
    )

    assert np.isfinite(
        prefix_scores
    ).all()

    run = scores_to_run(
        prefix_scores,
        query_ids=
            PREFIX_QUERY_IDS[split],
        doc_ids=
            DOC_IDS,
        depth=
            RUN_DEPTH
    )

    save_run(
        run,
        "prefix_minilm",
        split
    )

    PREFIX_RUNS[split] = run

    print(
        f"[{split}] "
        f"score matrix: "
        f"{prefix_scores.shape}"
    )

    print(
        f"[{split}] "
        f"saved top-{RUN_DEPTH} rankings"
    )


print(
    "\nPrefix MiniLM run status:"
)

display(
    run_status()
)

In [ ]:
def _sync_device(device):
    if device == "cuda":
        torch.cuda.synchronize()

    elif (
        device == "mps"
        and hasattr(torch, "mps")
        and hasattr(torch.mps, "synchronize")
    ):
        torch.mps.synchronize()


latency_texts = PREFIX_QUERY_TEXTS["test"]


# ============================================================
# 1. FULL WARM-UP PASS
# ============================================================
# Run every test query once without timing.
# This helps remove one-off MPS/GPU initialization or
# compilation overhead before the real latency measurement.

print(
    f"Warming up Prefix MiniLM with "
    f"{len(latency_texts)} test queries..."
)

for text_q in latency_texts:

    q_vec = prefix_model.encode(
        [text_q],
        batch_size=1,
        show_progress_bar=False,
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype(
        np.float32,
        copy=False
    )

    _ = q_vec @ prefix_doc_emb.T


_sync_device(PREFIX_DEVICE)

print("Warm-up complete.")


# ============================================================
# 2. MEASURE ONLINE QUERY LATENCY
# ============================================================
# Measures:
# query encoding
# +
# cosine scoring against all cached Prefix MiniLM documents

latencies_ms = []

for text_q in latency_texts:

    _sync_device(PREFIX_DEVICE)

    start = time.perf_counter()

    q_vec = prefix_model.encode(
        [text_q],
        batch_size=1,
        show_progress_bar=False,
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype(
        np.float32,
        copy=False
    )

    _scores = q_vec @ prefix_doc_emb.T

    _sync_device(PREFIX_DEVICE)

    elapsed_ms = (
        time.perf_counter() - start
    ) * 1000.0

    latencies_ms.append(
        elapsed_ms
    )


# ============================================================
# 3. LATENCY STATISTICS
# ============================================================

mean_latency_ms = float(
    np.mean(latencies_ms)
)

median_latency_ms = float(
    np.median(latencies_ms)
)

p95_latency_ms = float(
    np.percentile(
        latencies_ms,
        95
    )
)

max_latency_ms = float(
    np.max(latencies_ms)
)


# ============================================================
# 4. STORAGE / EMBEDDING COUNT
# ============================================================

storage_bytes = int(
    PREFIX_DOC_EMB_PATH
    .stat()
    .st_size
)

n_embeddings = int(
    prefix_doc_emb.shape[0]
)


# ============================================================
# 5. SAVE EFFICIENCY RESULTS
# ============================================================

prefix_efficiency = log_efficiency(
    "prefix_minilm",

    indexing_time_s=
        float(prefix_indexing_time_s),

    n_embeddings=
        n_embeddings,

    storage_bytes=
        storage_bytes,

    mean_latency_ms=
        mean_latency_ms,

    median_latency_ms=
        median_latency_ms,

    p95_latency_ms=
        p95_latency_ms,

    max_latency_ms=
        max_latency_ms,

    embedding_dim=
        int(PREFIX_EMB_DIM),

    dtype=
        str(prefix_doc_emb.dtype),

    max_seq_length=
        int(MAX_SEQ_LEN),

    batch_size=
        int(PREFIX_BATCH_SIZE),

    device=
        PREFIX_DEVICE,

    latency_queries=
        len(latency_texts),

    warmup_queries=
        len(latency_texts)
)


# ============================================================
# 6. DISPLAY RESULTS
# ============================================================

print(
    json.dumps(
        prefix_efficiency,
        indent=2
    )
)

print(
    "\nPrefix MiniLM efficiency\n"
)

print(
    f"Mean warm query latency: "
    f"{mean_latency_ms:.3f} ms"
)

print(
    f"Median warm query latency: "
    f"{median_latency_ms:.3f} ms"
)

print(
    f"P95 warm query latency: "
    f"{p95_latency_ms:.3f} ms"
)

print(
    f"Maximum query latency: "
    f"{max_latency_ms:.3f} ms"
)

print(
    f"Indexing time: "
    f"{prefix_indexing_time_s:.3f} s"
)

print(
    f"Document embeddings: "
    f"{n_embeddings:,}"
)

print(
    f"Embedding storage: "
    f"{storage_bytes / (1024**2):.2f} MiB"
)

In [ ]:
def recall_at_k(run, qrels, k=10):
    values = []

    for qid, relevant in qrels.items():

        top_docs = (
            run[run["query_id"] == qid]
            .sort_values("rank")
            .head(k)["doc_id"]
            .tolist()
        )

        relevant_docs = set(relevant.keys())

        recall = (
            len(set(top_docs) & relevant_docs)
            / len(relevant_docs)
        )

        values.append(recall)

    return np.mean(values)


def ndcg_at_k(run, qrels, k=10):
    values = []

    for qid, relevant in qrels.items():

        top_docs = (
            run[run["query_id"] == qid]
            .sort_values("rank")
            .head(k)["doc_id"]
            .tolist()
        )

        relevant_docs = set(relevant.keys())

        dcg = 0.0

        for i, doc_id in enumerate(top_docs, start=1):
            if doc_id in relevant_docs:
                dcg += 1 / np.log2(i + 1)

        ideal_n = min(len(relevant_docs), k)

        idcg = sum(
            1 / np.log2(i + 1)
            for i in range(1, ideal_n + 1)
        )

        values.append(
            dcg / idcg if idcg > 0 else 0.0
        )

    return np.mean(values)


prefix_test_run = load_run(
    "prefix_minilm",
    "test"
)

prefix_recall10 = recall_at_k(
    prefix_test_run,
    QRELS["test"],
    10
)

prefix_ndcg10 = ndcg_at_k(
    prefix_test_run,
    QRELS["test"],
    10
)

print("Prefix MiniLM Recall@10:", prefix_recall10)
print("Prefix MiniLM nDCG@10:", prefix_ndcg10)

---
# 5 · Windowed MiniLM  *(Member 5)*

Extension: split each `doc_text` into overlapping windows that each fit in MiniLM's limit, embed every window, and score a document by its **best window** (MaxP: S_window(q,d) = max over windows of cos(e_q, e_w)).

**Inputs (from Checkpoint A / disk)**
- `DOC_IDS`, `DOC_TEXTS`, `MODEL_NAME`, `MAX_SEQ_LEN`
- Query embeddings from Section 4 (`cache/query_emb__{split}.npy`); re-use them rather than re-encoding

**Required outputs**
- `data/processed/windows.parquet`: `doc_id, window_id, char_start, char_end, n_wordpieces` (**char spans in `doc_text`** are needed by Section 6.2)
- Cached window embeddings `cache/window_emb.npy` (+ row → (doc_id, window_id) map)
- `save_run(run, "window_minilm", split)` for train and test
- `log_efficiency("window_minilm", indexing_time_s=..., n_embeddings=..., storage_bytes=..., mean_latency_ms=..., p95_latency_ms=...)`

**Notes & checklist**
- [✅] Window size ≤ `MAX_SEQ_LEN - 2` content word-pieces; choose a stride (overlap) and record it in the config below
- [✅] Build windows from **tokenizer offsets** so the char spans are exact (`return_offsets_mapping=True`)
- [✅] Window construction must **not** use evidence locations or test labels
- [✅] Documents that fit in one window should get exactly one window (identical to the prefix embedding; good sanity check)
- [✅] Report embedding count and storage against prefix (5,183 embeddings)

In [ ]:
# 5.0 Window config (final choice recorded here)
# Windows hold at most MAX_SEQ_LEN - 2 = 254 content word-pieces (the same budget the prefix model gets),
# and advance by half a window, so every position is seen in two windows (50% overlap).
# The first window of every document is therefore identical to the prefix input.
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer

WINDOW_SIZE = MAX_SEQ_LEN - 2     # content word-pieces per window
STRIDE      = WINDOW_SIZE // 2    # step between window starts (127)
WINDOW_BATCH_SIZE = 64
assert 0 < STRIDE <= WINDOW_SIZE <= MAX_SEQ_LEN - 2

WINDOWS_PATH     = PROC_DIR / "windows.parquet"
WINDOW_RETENTION_PATH = PROC_DIR / "window_retention.parquet"
WINDOW_EMB_PATH  = CACHE_DIR / "window_emb.npy"
WINDOW_MAP_PATH  = CACHE_DIR / "window_emb_map.json"     # row i of window_emb.npy -> [doc_id, window_id]
WINDOW_META_PATH = CACHE_DIR / "window_emb.meta.json"

if torch.cuda.is_available():
    WINDOW_DEVICE = "cuda"
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    WINDOW_DEVICE = "mps"
else:
    WINDOW_DEVICE = "cpu"

window_tok = AutoTokenizer.from_pretrained(os.environ.get("LIBBOT_TOKENIZER", MODEL_NAME), use_fast=True)
assert window_tok.is_fast, "need a fast tokenizer for offset mapping"

window_model = SentenceTransformer(MODEL_NAME, device=WINDOW_DEVICE)
window_model.max_seq_length = MAX_SEQ_LEN          # same frozen model and limit as Section 4
assert window_model.max_seq_length == MAX_SEQ_LEN
window_model.eval()
for _p in window_model.parameters():
    _p.requires_grad_(False)
WINDOW_EMB_DIM = window_model.get_sentence_embedding_dimension()
print(f"window={WINDOW_SIZE} word-pieces, stride={STRIDE} | device={WINDOW_DEVICE} | model={MODEL_NAME}")

In [ ]:
# 5.1 Build windows with tokenizer offsets -> data/processed/windows.parquet
# Uses only document text (no evidence locations, no labels). Windows are listed in DOC_IDS order,
# so all windows of one document are contiguous (5.3 relies on this).
t0 = time.perf_counter()
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    _enc = window_tok(DOC_TEXTS, add_special_tokens=False, truncation=False,
                      return_offsets_mapping=True, verbose=False)

win_rows, WINDOW_IDS = [], []      # WINDOW_IDS: [CLS] + exact word-piece ids + [SEP] for each window
for doc_id, offs, ids in zip(DOC_IDS, _enc["offset_mapping"], _enc["input_ids"]):
    n = len(offs)
    start, wid = 0, 0
    while True:
        end = min(start + WINDOW_SIZE, n)
        win_rows.append((doc_id, wid, int(offs[start][0]), int(offs[end - 1][1]), end - start, start, end))
        WINDOW_IDS.append([window_tok.cls_token_id] + ids[start:end] + [window_tok.sep_token_id])
        if end >= n:
            break
        start += STRIDE
        wid += 1
windows = pd.DataFrame(win_rows, columns=["doc_id", "window_id", "char_start", "char_end",
                                          "n_wordpieces", "tok_start", "tok_end"])
windows.to_parquet(WINDOWS_PATH, index=False)

n_win_per_doc = windows.groupby("doc_id", sort=False).size()
_doc_wp = pd.Series([len(o) for o in _enc["offset_mapping"]], index=DOC_IDS)
print(f"built {len(windows):,} windows for {len(DOC_IDS):,} docs in {time.perf_counter() - t0:.1f}s "
      f"| mean {n_win_per_doc.mean():.2f}, max {n_win_per_doc.max()} windows per doc")
check("docs that fit in one window get exactly one window",
      (n_win_per_doc[_doc_wp.le(WINDOW_SIZE).values] == 1).all(), hard=True)
check("every window has <= WINDOW_SIZE word-pieces", (windows.n_wordpieces <= WINDOW_SIZE).all(), hard=True)
check("windows of a document are contiguous and in DOC_IDS order",
      list(n_win_per_doc.index) == DOC_IDS, hard=True)
check("last window of every document reaches the end of the text",
      (windows.groupby("doc_id", sort=False).tok_end.max().values == _doc_wp.values).all(), hard=True)
windows.head()

In [ ]:
# 5.2 Encode/cache window embeddings (time it)
# Windows are encoded from their exact word-piece ids (not from re-tokenised substrings): a window can start in the
# middle of a word, and re-tokenising that fragment would change the word-pieces (and the window length).
assert max(map(len, WINDOW_IDS)) <= MAX_SEQ_LEN
_hf = window_model[0].auto_model

@torch.no_grad()
def encode_window_ids(id_lists, batch_size=WINDOW_BATCH_SIZE, progress=True):
    """Same pipeline as SentenceTransformer.encode (mean pooling + L2 norm), fed with ready-made token ids."""
    from tqdm.auto import tqdm
    order = np.argsort([-len(x) for x in id_lists], kind="stable")     # longest first, like sentence-transformers
    out = np.zeros((len(id_lists), WINDOW_EMB_DIM), dtype=np.float32)
    for b in tqdm(range(0, len(order), batch_size), disable=not progress):
        idx = order[b:b + batch_size]
        L = max(len(id_lists[i]) for i in idx)
        ids = torch.full((len(idx), L), window_tok.pad_token_id, dtype=torch.long)
        mask = torch.zeros((len(idx), L), dtype=torch.long)
        for r, i in enumerate(idx):
            ids[r, :len(id_lists[i])] = torch.tensor(id_lists[i]); mask[r, :len(id_lists[i])] = 1
        ids, mask = ids.to(WINDOW_DEVICE), mask.to(WINDOW_DEVICE)
        h = _hf(input_ids=ids, attention_mask=mask).last_hidden_state
        m = mask.unsqueeze(-1).to(h.dtype)
        e = torch.nn.functional.normalize((h * m).sum(1) / m.sum(1).clamp(min=1e-9), dim=1)
        out[idx] = e.float().cpu().numpy()
    return out

expected_win_meta = {"model_name": MODEL_NAME, "max_seq_length": MAX_SEQ_LEN, "normalize_embeddings": True,
                     "embedding_dim": WINDOW_EMB_DIM, "n_windows": len(windows),
                     "window_size": WINDOW_SIZE, "stride": STRIDE}
win_keys = [[d, int(w)] for d, w in zip(windows.doc_id, windows.window_id)]

def _window_cache_ok():
    if not all(p.exists() for p in [WINDOW_EMB_PATH, WINDOW_MAP_PATH, WINDOW_META_PATH]):
        return False
    try:
        meta = json.loads(WINDOW_META_PATH.read_text(encoding="utf-8"))
        arr = np.load(WINDOW_EMB_PATH, mmap_mode="r")
        return (json.loads(WINDOW_MAP_PATH.read_text(encoding="utf-8")) == win_keys
                and all(meta.get(k) == v for k, v in expected_win_meta.items())
                and "indexing_time_s" in meta
                and arr.shape == (len(windows), WINDOW_EMB_DIM) and arr.dtype == np.float32)
    except Exception:
        return False

if _window_cache_ok():
    window_emb = np.load(WINDOW_EMB_PATH)
    window_indexing_time_s = float(json.loads(WINDOW_META_PATH.read_text(encoding="utf-8"))["indexing_time_s"])
    print("loaded cached window embeddings")
else:
    print(f"encoding {len(WINDOW_IDS):,} windows ...")
    t0 = time.perf_counter()
    window_emb = encode_window_ids(WINDOW_IDS)
    window_indexing_time_s = time.perf_counter() - t0
    np.save(WINDOW_EMB_PATH, window_emb)
    WINDOW_MAP_PATH.write_text(json.dumps(win_keys), encoding="utf-8")
    WINDOW_META_PATH.write_text(json.dumps({**expected_win_meta, "indexing_time_s": window_indexing_time_s,
                                            "batch_size": WINDOW_BATCH_SIZE, "device": WINDOW_DEVICE}, indent=2),
                                encoding="utf-8")
    print("saved window embeddings")

assert window_emb.shape == (len(windows), WINDOW_EMB_DIM) and window_emb.dtype == np.float32
assert np.allclose(np.linalg.norm(window_emb, axis=1), 1.0, atol=1e-4)

# sanity: a document with a single window must embed like the prefix model's embedding of that document
_prefix_path = CACHE_DIR / "prefix_doc_emb.npy"
if _prefix_path.exists():
    _pref = np.load(_prefix_path, mmap_mode="r")
    _single = np.flatnonzero((windows.window_id == 0).values & (n_win_per_doc.reindex(windows.doc_id).values == 1))
    _rows = np.array([CORPUS_IDX[d] for d in windows.doc_id.values[_single]])
    _cos = (window_emb[_single] * np.asarray(_pref)[_rows]).sum(axis=1)
    check("single-window docs match the prefix embedding", _cos.min() > 0.999, f"min cosine = {_cos.min():.6f}")
else:
    print("cache/prefix_doc_emb.npy not found — skipped the single-window vs prefix check (run Section 4 first)")
print(f"{len(windows):,} window embeddings | {WINDOW_EMB_PATH.stat().st_size / 2**20:.1f} MiB | "
      f"indexing {window_indexing_time_s:.1f}s | vs prefix: {len(windows) / len(DOC_IDS):.2f}x embeddings")

In [ ]:
# 5.3 MaxP aggregation -> doc score matrix -> scores_to_run -> save_run('window_minilm', split)
# S_window(q, d) = max over the windows of d of cos(e_q, e_w). Embeddings are unit length, so dot = cosine.
DOC_FIRST_WIN = np.flatnonzero((windows.window_id == 0).values)        # first window row of each doc, DOC_IDS order
assert len(DOC_FIRST_WIN) == len(DOC_IDS)

def maxp(q_emb):
    """(n_q x 384) query embeddings -> (n_q x n_docs) best-window cosine."""
    return np.maximum.reduceat(q_emb @ window_emb.T, DOC_FIRST_WIN, axis=1)

def load_query_emb(split):
    """Section 4's cached claim embeddings if they match this split's queries, else encode them here (same model)."""
    qids = Q[split]["query_id"].astype(str).tolist()
    p, pi = CACHE_DIR / f"query_emb__{split}.npy", CACHE_DIR / f"query_ids__{split}.json"
    if p.exists() and pi.exists() and [str(x) for x in json.loads(pi.read_text(encoding="utf-8"))] == qids:
        arr = np.load(p)
        if arr.shape == (len(qids), WINDOW_EMB_DIM):
            return qids, arr.astype(np.float32, copy=False)
    arr = window_model.encode(Q[split]["claim"].astype(str).tolist(), batch_size=WINDOW_BATCH_SIZE,
                              convert_to_numpy=True, normalize_embeddings=True)
    return qids, arr.astype(np.float32, copy=False)

WINDOW_RUNS = {}
for split in SPLITS:
    qids, q_emb = load_query_emb(split)
    doc_scores = maxp(q_emb)
    assert doc_scores.shape == (len(qids), len(DOC_IDS)) and np.isfinite(doc_scores).all()
    WINDOW_RUNS[split] = scores_to_run(doc_scores, query_ids=qids, doc_ids=DOC_IDS, depth=RUN_DEPTH)
    save_run(WINDOW_RUNS[split], "window_minilm", split)
run_status()

In [ ]:
# 5.4 Warm latency, storage, embedding count -> log_efficiency('window_minilm', ...)
# Latency per query = encode the claim + cosine against every window + max over each doc's windows,
# measured after a full warm-up pass (same protocol as prefix_minilm in Section 4).
def _sync(device):
    if device == "cuda":
        torch.cuda.synchronize()
    elif device == "mps" and hasattr(torch, "mps") and hasattr(torch.mps, "synchronize"):
        torch.mps.synchronize()

def _one_query(text):
    v = window_model.encode([text], batch_size=1, show_progress_bar=False,
                            convert_to_numpy=True, normalize_embeddings=True).astype(np.float32, copy=False)
    return maxp(v)

lat_texts = Q["test"]["claim"].astype(str).tolist()
for t in lat_texts:                      # warm-up (not timed)
    _one_query(t)
_sync(WINDOW_DEVICE)

lat_ms = []
for t in lat_texts:
    _sync(WINDOW_DEVICE)
    t0 = time.perf_counter()
    _one_query(t)
    _sync(WINDOW_DEVICE)
    lat_ms.append((time.perf_counter() - t0) * 1000.0)

window_efficiency = log_efficiency(
    "window_minilm",
    indexing_time_s=float(window_indexing_time_s),
    n_embeddings=int(window_emb.shape[0]),
    storage_bytes=int(WINDOW_EMB_PATH.stat().st_size),
    mean_latency_ms=float(np.mean(lat_ms)),
    median_latency_ms=float(np.median(lat_ms)),
    p95_latency_ms=float(np.percentile(lat_ms, 95)),
    max_latency_ms=float(np.max(lat_ms)),
    embedding_dim=int(WINDOW_EMB_DIM), dtype=str(window_emb.dtype),
    max_seq_length=int(MAX_SEQ_LEN), window_size=int(WINDOW_SIZE), stride=int(STRIDE),
    windows_per_doc_mean=float(n_win_per_doc.mean()),
    batch_size=int(WINDOW_BATCH_SIZE), device=WINDOW_DEVICE,
    latency_queries=len(lat_texts), warmup_queries=len(lat_texts))
print(json.dumps(window_efficiency, indent=2))

---
# 6 · Truncation and evidence-retention diagnostic  *(Members 2 + 5)*

*Can the model see the evidence?* For each test (claim, relevant doc) pair with annotated evidence, we check whether the evidence characters fall inside the part of `doc_text` that the prefix model actually encodes.

| Step (proposal) | Where | Owner |
|---|---|---|
| 1 · Tokenisation and truncation tracking: tokenizer offsets → retained char span | 6.1 | M2 ✅ |
| 2 · Evidence localisation: char spans of the expert evidence sentences | Section 2.6 | M2 ✅ |
| 3 · Retention classification: complete / partial / absent | 6.1 | M2 ✅ |
| Window coverage of evidence | 6.2 | M5 |
| 4 · Recovery by visibility group, group sizes, gains/regressions, diagnostic records | 6.3 | M2 + M5 |

> **Caveat (keep in the report):** MiniLM truncation is a hypothesis to audit. Missing evidence does not prove that truncation caused a ranking failure.

### 6.1 Prefix retention  *(Member 2)*

- Tokenise every `doc_text` with the MiniLM tokenizer (offsets on, no truncation). The model keeps `MAX_SEQ_LEN − 2 = 254` content word-pieces, so `retained_char_end` is the end offset of the 254th word-piece, or the full length if the document is shorter.
- For each pair: `retained_frac` = share of evidence characters before `retained_char_end`.
  **complete** = 1.0 · **partial** = between 0 and 1 · **absent** = 0. The calculation uses the union of all rationale sets. `any_set_complete` records whether at least one full rationale set is visible.
- Computed for both splits so the diagnostic is available everywhere, but only **test** is reported, and train is never used to pick window settings.

In [ ]:
from transformers import AutoTokenizer
TOKENIZER_NAME = os.environ.get("LIBBOT_TOKENIZER", MODEL_NAME)   # env override only for offline testing
tok = AutoTokenizer.from_pretrained(TOKENIZER_NAME, use_fast=True)
assert tok.is_fast, "need a fast tokenizer for offset mapping"
N_SPECIAL = tok.num_special_tokens_to_add(pair=False)
CONTENT_BUDGET = MAX_SEQ_LEN - N_SPECIAL
print(f"tokenizer={TOKENIZER_NAME} | special tokens={N_SPECIAL} | content budget={CONTENT_BUDGET} word-pieces")

corpus, qrel_pairs, evidence_sentences = load_processed("corpus", "qrel_pairs", "evidence_sentences")

t0 = time.perf_counter()
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    enc = tok(corpus.doc_text.tolist(), add_special_tokens=False, truncation=False,
              return_offsets_mapping=True, verbose=False)
n_wp, cut = [], []
for text, offs in zip(corpus.doc_text, enc["offset_mapping"]):
    n_wp.append(len(offs))
    cut.append(len(text) if len(offs) <= CONTENT_BUDGET else int(offs[CONTENT_BUDGET - 1][1]))
print(f"tokenised {len(corpus)} docs in {time.perf_counter() - t0:.1f}s")

doc_stats = pd.DataFrame({"doc_id": corpus.doc_id, "n_wordpieces": n_wp, "n_chars": corpus.n_chars,
                          "retained_char_end": cut})
doc_stats["truncated"] = doc_stats.n_wordpieces > CONTENT_BUDGET
doc_stats["retained_char_frac"] = doc_stats.retained_char_end / doc_stats.n_chars.clip(lower=1)
doc_stats["n_sentences_fully_retained"] = [sum(e <= c for e in ends if e >= 0)
                                           for ends, c in zip(corpus.sent_end, cut)]

# cross-check: our cutoff matches what the model really receives with truncation on
sample = corpus.sample(min(200, len(corpus)), random_state=SEED)
real_len = [len(x) for x in tok(sample.doc_text.tolist(), truncation=True, max_length=MAX_SEQ_LEN)["input_ids"]]
ours = [min(n, CONTENT_BUDGET) + N_SPECIAL for n in doc_stats.set_index("doc_id").loc[sample.doc_id, "n_wordpieces"]]
check("retained length matches tokenizer(truncation=True, max_length=MAX_SEQ_LEN)", real_len == ours, hard=True)
n_title_cut = sum(c < len(t) for c, t in zip(cut, corpus.title))
check("truncation never cuts before the title ends", n_title_cut == 0, f"{n_title_cut} docs cut inside the title")
doc_stats.describe().round(2)

In [ ]:
CUT = dict(zip(doc_stats.doc_id, doc_stats.retained_char_end))

def retention(starts, ends, cutoff):
    total = sum(e - s for s, e in zip(starts, ends))
    kept = sum(max(0, min(e, cutoff) - s) for s, e in zip(starts, ends))
    full = sum(e <= cutoff for e in ends)
    none = sum(s >= cutoff for s in starts)
    return total, kept, full, len(starts) - full - none, none

# last evidence char of each rationale set, per (claim, doc)
SET_END = (evidence_sentences.groupby(["query_id", "doc_id", "rationale_set"]).char_end.max()
           .groupby(level=[0, 1]).apply(list).to_dict())

ret_rows = []
for p in qrel_pairs[qrel_pairs.has_evidence & qrel_pairs.spans_ok].itertuples(index=False):
    c = CUT[p.doc_id]
    total, kept, full, part, none = retention(p.ev_char_start, p.ev_char_end, c)
    frac = kept / total if total else np.nan
    any_set_complete = any(e <= c for e in SET_END.get((p.query_id, p.doc_id), []))
    ret_rows.append({"query_id": p.query_id, "doc_id": p.doc_id, "split": p.split, "label": p.label,
                     "retained_char_end": c, "doc_n_chars": p.doc_n_chars,
                     "n_evidence_sentences": p.n_evidence_sentences,
                     "ev_sent_full": full, "ev_sent_partial": part, "ev_sent_absent": none,
                     "retained_frac": frac,
                     "visibility": "complete" if frac >= 1 else ("absent" if frac <= 0 else "partial"),
                     "any_set_complete": any_set_complete})
prefix_retention = pd.DataFrame(ret_rows)
prefix_retention["doc_truncated"] = prefix_retention.doc_id.map(dict(zip(doc_stats.doc_id, doc_stats.truncated)))

doc_stats.to_parquet(PROCESSED_FILES["doc_token_stats"], index=False)
prefix_retention.to_parquet(PROCESSED_FILES["prefix_retention"], index=False)

order = ["complete", "partial", "absent"]
tab = pd.crosstab(prefix_retention.split, prefix_retention.visibility).reindex(index=SPLITS, columns=order, fill_value=0)
tab_pct = tab.div(tab.sum(axis=1), axis=0).mul(100).round(1).add_suffix(" %")
print(f"Docs longer than the prefix window: {doc_stats.truncated.sum()} / {len(doc_stats)} ({doc_stats.truncated.mean():.1%})")
display(pd.concat([tab, tab_pct], axis=1))

t = prefix_retention[prefix_retention.split == "test"]
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
axes[0].hist(doc_stats.n_wordpieces, bins=60, color="#5b7553")
axes[0].axvline(CONTENT_BUDGET, color="crimson", ls="--", label=f"cutoff ({CONTENT_BUDGET} content word-pieces)")
axes[0].set(title="Document length in MiniLM word-pieces", xlabel="word-pieces", ylabel="documents"); axes[0].legend()
counts = t.visibility.value_counts().reindex(order, fill_value=0)
axes[1].bar(counts.index, counts.values, color=["#5b7553", "#c9a227", "#a23b3b"])
for i, v in enumerate(counts.values):
    axes[1].text(i, v, f"{v} ({v / max(len(t), 1):.0%})", ha="center", va="bottom")
axes[1].set(title="Test pairs: evidence visible to prefix MiniLM?", ylabel="(claim, doc) pairs")
plt.tight_layout(); fig.savefig(FIG_DIR / "s6_prefix_retention.png", dpi=150); plt.show()

### 6.2 Window coverage of evidence  *(Member 5)*

**Inputs:** `windows.parquet` (Section 5) and `prefix_retention` / `qrel_pairs` (above).
**Output:** add `window_full_cover` (some single window contains all the evidence chars) and `window_any_cover` to a copy of `prefix_retention` → `data/processed/window_retention.parquet`.

In [ ]:
# 6.2 Evidence coverage by windows -> data/processed/window_retention.parquet
# Evidence = union of all annotated rationale sentences of a (claim, doc) pair (same definition as 6.1).
#   window_full_cover   : one single window contains ALL evidence characters (first evidence start .. last evidence end)
#   window_any_cover    : one single window contains at least one complete evidence sentence
#   window_best_frac    : largest share of the evidence characters that fall inside any one window
#   window_any_set_cover: one single window contains a complete rationale set (cf. any_set_complete in 6.1)
windows = pd.read_parquet(WINDOWS_PATH)
prefix_retention, qrel_pairs, evidence_sentences = load_processed("prefix_retention", "qrel_pairs", "evidence_sentences")
spans = qrel_pairs.set_index(["query_id", "doc_id"])[["ev_char_start", "ev_char_end"]]
WIN_BY_DOC = {d: (g.char_start.to_numpy(), g.char_end.to_numpy()) for d, g in windows.groupby("doc_id")}
SET_SPAN = (evidence_sentences.groupby(["query_id", "doc_id", "rationale_set"])
            .agg(s=("char_start", "min"), e=("char_end", "max")).reset_index()
            .groupby(["query_id", "doc_id"]).apply(lambda g: list(zip(g.s, g.e)), include_groups=False).to_dict())

full, anyc, best, anyset = [], [], [], []
for r in prefix_retention.itertuples(index=False):
    ev_s, ev_e = (np.asarray(x) for x in spans.loc[(r.query_id, r.doc_id)])
    ws, we = WIN_BY_DOC[r.doc_id]
    inside = (ws[:, None] <= ev_s[None, :]) & (we[:, None] >= ev_e[None, :])          # window x sentence: fully inside
    full.append(bool(((ws <= ev_s.min()) & (we >= ev_e.max())).any()))
    anyc.append(bool(inside.any()))
    kept = np.clip(np.minimum(we[:, None], ev_e[None, :]) - np.maximum(ws[:, None], ev_s[None, :]), 0, None).sum(axis=1)
    best.append(float(kept.max() / (ev_e - ev_s).sum()))
    anyset.append(any(((ws <= s) & (we >= e)).any() for s, e in SET_SPAN.get((r.query_id, r.doc_id), [])))

window_retention = prefix_retention.copy()
window_retention["window_full_cover"] = full
window_retention["window_any_cover"] = anyc
window_retention["window_best_frac"] = best
window_retention["window_any_set_cover"] = anyset
window_retention.to_parquet(WINDOW_RETENTION_PATH, index=False)

check("a window that fully covers the evidence implies best_frac = 1",
      (window_retention.loc[window_retention.window_full_cover, "window_best_frac"] > 1 - 1e-9).all(), hard=True)
check("pairs fully visible to the prefix are also fully covered by a window",
      window_retention.loc[window_retention.visibility == "complete", "window_full_cover"].all())

t = window_retention[window_retention.split == "test"]
tab = (t.groupby("visibility")
       .agg(n=("query_id", "size"), window_full_cover=("window_full_cover", "mean"),
            window_any_cover=("window_any_cover", "mean"), window_best_frac=("window_best_frac", "mean"))
       .reindex(["complete", "partial", "absent"]))
print(f"Test pairs with evidence: {len(t)} | fully covered by one window: {t.window_full_cover.mean():.1%} "
      f"(prefix sees all of it: {(t.visibility == 'complete').mean():.1%})")
display(tab.round(3))

### 6.3 Recovery by visibility group, gains and regressions  *(Members 2 + 5, Week 3)*

**Inputs:** `prefix_retention` (test), runs `prefix_minilm__test`, `window_minilm__test` (+ BM25 and hybrids for context).
**Plan:**
1. For each test pair, compute rank of the relevant doc under each method → `hit@10`.
2. Table: hit@10 by visibility group (complete / partial / absent) × method, **with group sizes** (n per group).
3. Gains and regressions: pairs where windowed hits@10 and prefix misses (gain), and the reverse (regression). List claim, doc title and evidence sentence for inspection.
4. Save `results/metrics/diagnostic_records.parquet` (one row per test pair × method).

In [ ]:
# 6.3 Build one record per test qrel pair and retrieval method
prefix_retention = pd.read_parquet(PROC_DIR / "prefix_retention.parquet")
window_retention = pd.read_parquet(PROC_DIR / "window_retention.parquet")
test_pairs = pd.read_parquet(PROC_DIR / "qrel_pairs.parquet").query("split == 'test'").copy()
test_pairs["query_id"] = test_pairs["query_id"].astype(str)
test_pairs["doc_id"] = test_pairs["doc_id"].astype(str)
prefix_cols = ["query_id", "doc_id", "visibility", "ev_sent_full", "ev_sent_partial", "ev_sent_absent", "retained_frac"]
window_cols = ["query_id", "doc_id", "window_full_cover", "window_any_cover", "window_best_frac", "window_any_set_cover"]
test_pairs = test_pairs.merge(prefix_retention[prefix_cols], on=["query_id", "doc_id"], how="left", validate="one_to_one")
test_pairs = test_pairs.merge(window_retention[window_cols], on=["query_id", "doc_id"], how="left", validate="one_to_one")
test_pairs["visibility"] = test_pairs["visibility"].fillna("no_evidence")

corpus_titles = pd.read_parquet(PROC_DIR / "corpus.parquet", columns=["doc_id", "title"])
test_pairs = test_pairs.merge(corpus_titles, on="doc_id", how="left", validate="many_to_one")
evidence = pd.read_parquet(PROC_DIR / "evidence_sentences.parquet").query("split == 'test'")
evidence_text = (evidence.sort_values(["query_id", "doc_id", "sentence_idx"])
                 .groupby(["query_id", "doc_id"])["sentence_text"]
                 .agg(lambda rows: " ".join(dict.fromkeys(rows)))
                 .rename("evidence_text").reset_index())
test_pairs = test_pairs.merge(evidence_text, on=["query_id", "doc_id"], how="left", validate="one_to_one")
test_pairs["evidence_text"] = test_pairs["evidence_text"].fillna("")

diagnostic_parts = []
for method in METHODS:
    ranking = load_run(method, "test")[["query_id", "doc_id", "rank"]].copy()
    ranking["query_id"] = ranking["query_id"].astype(str)
    ranking["doc_id"] = ranking["doc_id"].astype(str)
    part = test_pairs.merge(ranking, on=["query_id", "doc_id"], how="left", validate="one_to_one")
    part["method"] = method
    part["hit@10"] = part["rank"].le(K)
    diagnostic_parts.append(part)
diagnostic_records = pd.concat(diagnostic_parts, ignore_index=True)
assert len(diagnostic_records) == len(test_pairs) * len(METHODS)
assert diagnostic_records.groupby("method")["query_id"].size().eq(len(test_pairs)).all()
diagnostic_records.to_parquet(METRICS_DIR / "diagnostic_records.parquet", index=False)

evidence_records = diagnostic_records[diagnostic_records["has_evidence"].fillna(False)].copy()
visibility_order = ["complete", "partial", "absent"]
visibility_summary = (evidence_records.groupby(["visibility", "method"], observed=True)
                      .agg(n_pairs=("hit@10", "size"), hit_rate_at_10=("hit@10", "mean"))
                      .reset_index())
visibility_summary["visibility"] = pd.Categorical(visibility_summary["visibility"], categories=visibility_order, ordered=True)
visibility_summary = visibility_summary.sort_values(["visibility", "method"]).reset_index(drop=True)
visibility_summary.to_csv(METRICS_DIR / "diagnostic_by_visibility.csv", index=False)
display(visibility_summary)
print(f"Saved {len(diagnostic_records):,} records ({len(test_pairs):,} test pairs × {len(METHODS)} methods). No-evidence pairs are retained as a separate group.")

In [ ]:
# Windowed MiniLM gains / regressions versus prefix MiniLM
hit_comparison = (diagnostic_records[diagnostic_records["method"].isin(["prefix_minilm", "window_minilm"])]
                  .pivot(index=["query_id", "doc_id"], columns="method", values="hit@10")
                  .reset_index())
pair_details = test_pairs[["query_id", "doc_id", "claim", "title", "visibility", "has_evidence", "evidence_text"]]
gain_regression = hit_comparison.merge(pair_details, on=["query_id", "doc_id"], how="left", validate="one_to_one")
gain_regression["change"] = np.select(
    [gain_regression["window_minilm"] & ~gain_regression["prefix_minilm"],
     gain_regression["prefix_minilm"] & ~gain_regression["window_minilm"]],
    ["gain", "regression"], default="unchanged")
gain_regression = gain_regression[gain_regression["change"] != "unchanged"].sort_values(["change", "visibility", "query_id", "doc_id"])
gain_regression.to_parquet(METRICS_DIR / "window_prefix_gain_regression.parquet", index=False)
display(gain_regression[["change", "query_id", "claim", "doc_id", "title", "visibility", "has_evidence", "evidence_text"]].head(50))
print(gain_regression["change"].value_counts().rename_axis("change").to_frame("n_pairs"))
print("Saved gains/regressions for manual inspection; absent evidence text indicates a rationale-free relevant qrel pair.")

---
# 7 · Hybrid fusion and α tuning  *(Member 6)*

Per-query min-max normalisation of each method's scores, then $S_{hybrid}^{(r)} = \alpha_r \hat S_r + (1-\alpha_r)\hat S_{BM25}$, $r \in \{prefix, window\}$. Tune $\alpha$ on **train only**, freeze, then apply once to test.

**Inputs (from Checkpoint A / disk)**
- Runs: `load_run("bm25", split)`, `load_run("prefix_minilm", split)`, `load_run("window_minilm", split)` (top-`RUN_DEPTH` raw scores)
- `ALPHA_GRID` (same grid for both hybrids), `QRELS["train"]`

**Required outputs**
- `results/frozen_alphas.json`: `{"hybrid_prefix": α, "hybrid_window": α, "selected_by": "train nDCG@10", "grid": ALPHA_GRID}`
- `results/metrics/alpha_sweep.parquet` (method, α, train nDCG@10, Recall@10)
- `save_run(run, "hybrid_prefix", split)` and `save_run(run, "hybrid_window", split)` for train and test

**Notes & checklist**
- [✅] Candidate pool is the per-query union of both top-`RUN_DEPTH` runs; scores are min-max normalised within each run, and a missing candidate gets 0.
- [✅] Constant-score runs map to 0 without division by zero.
- [✅] Endpoint assertions verify α = 0 reproduces BM25 and α = 1 reproduces the semantic ranking.
- [✅] Select α using train metrics only; load the frozen weights before generating test rankings.

In [ ]:
# 7.1 Per-query min-max normalisation + fusion function
def minmax_score_map(run):
    """Normalize one query's retrieved scores; absent candidates receive 0 after fusion."""
    if run.duplicated(["query_id", "doc_id"]).any():
        raise ValueError("A run contains duplicate query/document rows")
    out = {}
    for qid, rows in run.groupby("query_id", sort=False):
        scores = rows["score"].to_numpy(dtype=np.float64)
        if not np.isfinite(scores).all():
            raise ValueError(f"Non-finite scores for query {qid}")
        lo, hi = scores.min(), scores.max()
        scaled = np.zeros_like(scores) if hi == lo else (scores - lo) / (hi - lo)
        out[str(qid)] = pd.Series(scaled, index=rows["doc_id"].astype(str)).to_dict()
    return out

def prepare_fusion(bm25_run, semantic_run):
    """Precompute normalized scores and candidate arrays once for a pair of runs."""
    bq = set(bm25_run["query_id"].astype(str))
    sq = set(semantic_run["query_id"].astype(str))
    if bq != sq:
        raise ValueError(f"Query sets differ: BM25-only={len(bq-sq)}, semantic-only={len(sq-bq)}")
    bnorm, snorm = minmax_score_map(bm25_run), minmax_score_map(semantic_run)
    b_groups = {str(qid): rows for qid, rows in bm25_run.groupby("query_id", sort=False)}
    s_groups = {str(qid): rows for qid, rows in semantic_run.groupby("query_id", sort=False)}
    prepared = {}
    for qid in sorted(bq):
        bdocs, sdocs = bnorm[qid], snorm[qid]
        doc_ids = np.asarray(sorted(set(bdocs) | set(sdocs)), dtype=object)
        b_rank = dict(zip(b_groups[qid]["doc_id"].astype(str), b_groups[qid]["rank"].astype(int)))
        s_rank = dict(zip(s_groups[qid]["doc_id"].astype(str), s_groups[qid]["rank"].astype(int)))
        prepared[qid] = {
            "doc_ids": doc_ids,
            "bm25_scores": np.fromiter((bdocs.get(d, 0.0) for d in doc_ids), dtype=np.float64, count=len(doc_ids)),
            "semantic_scores": np.fromiter((sdocs.get(d, 0.0) for d in doc_ids), dtype=np.float64, count=len(doc_ids)),
            "bm25_ranks": np.fromiter((b_rank.get(d, np.inf) for d in doc_ids), dtype=np.float64, count=len(doc_ids)),
            "semantic_ranks": np.fromiter((s_rank.get(d, np.inf) for d in doc_ids), dtype=np.float64, count=len(doc_ids)),
        }
    return prepared

def _fusion_order(data, alpha):
    if alpha == 0.0:
        scores, order = data["bm25_scores"], np.lexsort((data["doc_ids"], data["bm25_ranks"]))
    elif alpha == 1.0:
        scores, order = data["semantic_scores"], np.lexsort((data["doc_ids"], data["semantic_ranks"]))
    else:
        scores = (1.0 - alpha) * data["bm25_scores"] + alpha * data["semantic_scores"]
        order = np.lexsort((data["doc_ids"], data["semantic_ranks"], data["bm25_ranks"], -scores))
    return order, scores

def fuse_prepared_runs(prepared, alpha, depth=RUN_DEPTH):
    """Rank each query's candidate union; absent-from-run scores are normalized to zero."""
    if not 0.0 <= float(alpha) <= 1.0:
        raise ValueError(f"alpha must be in [0, 1], got {alpha}")
    parts = []
    for qid, data in prepared.items():
        order, scores = _fusion_order(data, float(alpha))
        order = order[:depth]
        parts.append(pd.DataFrame({"query_id": qid, "doc_id": data["doc_ids"][order], "score": scores[order], "rank": np.arange(1, len(order) + 1)}))
    return pd.concat(parts, ignore_index=True)

def fuse_runs(bm25_run, semantic_run, alpha, depth=RUN_DEPTH):
    """Fuse the union of two top-depth runs using per-query min-max normalized scores."""
    return fuse_prepared_runs(prepare_fusion(bm25_run, semantic_run), alpha, depth=depth)

def measure_fusion_grid(prepared, qrels, alphas=ALPHA_GRID, k=K):
    """Measure train metrics for each alpha without materializing full ranked runs."""
    metrics = {float(a): {"recall": [], "ndcg": []} for a in alphas}
    for qid, rel in qrels.items():
        data = prepared[qid]
        for alpha in metrics:
            order, _ = _fusion_order(data, alpha)
            top_docs = data["doc_ids"][order[:k]]
            hits = np.fromiter((float(doc in rel) for doc in top_docs), dtype=np.float64, count=len(top_docs))
            dcg = float(np.sum(hits / np.log2(np.arange(2, len(hits) + 2))))
            idcg = float(np.sum(1.0 / np.log2(np.arange(2, min(len(rel), k) + 2))))
            metrics[alpha]["recall"].append(float(hits.sum() / len(rel)))
            metrics[alpha]["ndcg"].append(dcg / idcg)
    return {alpha: {f"recall@{k}": float(np.mean(values["recall"])), f"ndcg@{k}": float(np.mean(values["ndcg"])), "n_queries": len(values["recall"])} for alpha, values in metrics.items()}

print("Hybrid fusion helpers ready; candidate pool is the union of the two top-RUN_DEPTH runs.")

In [ ]:
# 7.2 α sweep on train -> alpha_sweep.parquet, pick best by train nDCG@10, write frozen_alphas.json
HYBRID_BASES = {"hybrid_prefix": "prefix_minilm", "hybrid_window": "window_minilm"}
bm25_train = load_run("bm25", "train")
alpha_rows = []
for hybrid_method, semantic_method in HYBRID_BASES.items():
    semantic_train = load_run(semantic_method, "train")
    prepared_train = prepare_fusion(bm25_train, semantic_train)
    grid_metrics = measure_fusion_grid(prepared_train, QRELS["train"], ALPHA_GRID, k=K)
    for alpha in ALPHA_GRID:
        metrics = grid_metrics[float(alpha)]
        alpha_rows.append({"method": hybrid_method, "alpha": float(alpha), **metrics})
        print(f"{hybrid_method:<14} alpha={alpha:.2f}  train R@{K}={metrics[f'recall@{K}']:.4f}  nDCG@{K}={metrics[f'ndcg@{K}']:.4f}")

alpha_sweep = pd.DataFrame(alpha_rows)
alpha_sweep.to_parquet(METRICS_DIR / "alpha_sweep.parquet", index=False)
frozen_alphas = {}
for method in HYBRID_BASES:
    best = (alpha_sweep[alpha_sweep["method"] == method]
            .sort_values([f"ndcg@{K}", f"recall@{K}", "alpha"], ascending=[False, False, True], kind="mergesort")
            .iloc[0])
    frozen_alphas[method] = float(best["alpha"])
    print(f"Frozen {method} alpha={best['alpha']:.2f} by train nDCG@{K}={best[f'ndcg@{K}']:.4f}")

frozen_alphas.update({"selected_by": f"train nDCG@{K}", "grid": [float(a) for a in ALPHA_GRID]})
with open(ROOT / "results" / "frozen_alphas.json", "w", encoding="utf-8") as f:
    json.dump(frozen_alphas, f, indent=2)
print("Saved alpha_sweep.parquet and frozen_alphas.json")

In [ ]:
# 7.3 Apply frozen α to train + test -> save_run('hybrid_prefix'/'hybrid_window', split)
with open(ROOT / "results" / "frozen_alphas.json", encoding="utf-8") as f:
    frozen_alphas = json.load(f)

# Endpoint checks ensure normalization and missing-candidate handling preserve each source ranking.
for split in SPLITS:
    bm25_run = load_run("bm25", split)
    for hybrid_method, semantic_method in HYBRID_BASES.items():
        semantic_run = load_run(semantic_method, split)
        prepared = prepare_fusion(bm25_run, semantic_run)
        for endpoint, source in [(0.0, bm25_run), (1.0, semantic_run)]:
            actual = fuse_prepared_runs(prepared, endpoint)
            expected_pairs = source.sort_values(["query_id", "rank"])[["query_id", "doc_id"]].reset_index(drop=True)
            actual_pairs = actual.sort_values(["query_id", "rank"])[["query_id", "doc_id"]].reset_index(drop=True)
            assert actual_pairs.equals(expected_pairs), f"{hybrid_method}/{split}: alpha={endpoint} changed the source ranking"
        hybrid_run = fuse_prepared_runs(prepared, frozen_alphas[hybrid_method])
        save_run(hybrid_run, hybrid_method, split)

print("Saved both frozen hybrid rankings for train and test; alpha selection used train only.")

In [ ]:
# 7.4 Re-measure every method with one hardware, warm-up and repetition protocol
# Run after Sections 3–7: this cell needs the live BM25 index, both embedding models and cached document vectors.
latency_claims = Q["test"]["claim"].astype(str).tolist()
latency_doc_ids = np.asarray([str(doc_id) for doc_id in DOC_IDS], dtype=object)
LATENCY_REPEATS = 3

def _latency_top_indices(scores, depth):
    scores = np.asarray(scores, dtype=np.float64)
    if scores.shape != (len(DOC_IDS),) or not np.isfinite(scores).all():
        raise ValueError("Online scoring returned an invalid document score vector")
    depth = min(int(depth), len(scores))
    selected = np.argpartition(-scores, depth - 1)[:depth]
    return selected[np.lexsort((latency_doc_ids[selected], -scores[selected]))]

def _latency_minmax(scores):
    scores = np.asarray(scores, dtype=np.float64)
    lo, hi = scores.min(), scores.max()
    return np.zeros_like(scores) if hi == lo else (scores - lo) / (hi - lo)

def _latency_search(method, claim):
    if method == "bm25":
        bm25_scores = np.asarray(bm25.get_scores(bm25_tokenise(claim)), dtype=np.float64)
        return _latency_top_indices(bm25_scores, K)

    if method in {"prefix_minilm", "hybrid_prefix"}:
        query_vec = prefix_model.encode([claim], batch_size=1, show_progress_bar=False,
                                        convert_to_numpy=True, normalize_embeddings=True).astype(np.float32, copy=False)
        semantic_scores = (query_vec @ prefix_doc_emb.T).reshape(-1)
    elif method in {"window_minilm", "hybrid_window"}:
        query_vec = window_model.encode([claim], batch_size=1, show_progress_bar=False,
                                        convert_to_numpy=True, normalize_embeddings=True).astype(np.float32, copy=False)
        semantic_scores = np.maximum.reduceat(query_vec @ window_emb.T, DOC_FIRST_WIN, axis=1).reshape(-1)
    else:
        raise ValueError(f"Unknown method: {method}")

    if method in {"prefix_minilm", "window_minilm"}:
        return _latency_top_indices(semantic_scores, K)

    bm25_scores = np.asarray(bm25.get_scores(bm25_tokenise(claim)), dtype=np.float64)
    bm25_top = _latency_top_indices(bm25_scores, RUN_DEPTH)
    semantic_top = _latency_top_indices(semantic_scores, RUN_DEPTH)
    candidates = np.union1d(bm25_top, semantic_top)
    bm25_scores_by_doc = _latency_minmax(bm25_scores[bm25_top])
    semantic_scores_by_doc = _latency_minmax(semantic_scores[semantic_top])
    bm25_map = dict(zip(bm25_top.tolist(), bm25_scores_by_doc.tolist()))
    semantic_map = dict(zip(semantic_top.tolist(), semantic_scores_by_doc.tolist()))
    bm25_rank_map = {int(doc): rank for rank, doc in enumerate(bm25_top, start=1)}
    semantic_rank_map = {int(doc): rank for rank, doc in enumerate(semantic_top, start=1)}
    alpha = float(frozen_alphas[method])
    fused_scores = np.fromiter(((1.0 - alpha) * bm25_map.get(int(doc), 0.0)
                                + alpha * semantic_map.get(int(doc), 0.0) for doc in candidates),
                               dtype=np.float64, count=len(candidates))
    bm25_ranks = np.fromiter((bm25_rank_map.get(int(doc), np.inf) for doc in candidates), dtype=np.float64, count=len(candidates))
    semantic_ranks = np.fromiter((semantic_rank_map.get(int(doc), np.inf) for doc in candidates), dtype=np.float64, count=len(candidates))
    top_order = np.lexsort((latency_doc_ids[candidates], semantic_ranks, bm25_ranks, -fused_scores))[:K]
    return candidates[top_order]

latency_methods = ["bm25", "prefix_minilm", "window_minilm", "hybrid_prefix", "hybrid_window"]
latency_ms_by_method = {}
for method in latency_methods:
    print(f"Warming up {method} with all {len(latency_claims)} test claims...")
    for claim in latency_claims:
        _latency_search(method, claim)
    _sync_device(PREFIX_DEVICE)
    if WINDOW_DEVICE != PREFIX_DEVICE:
        _sync_device(WINDOW_DEVICE)

    timings = []
    for _ in range(LATENCY_REPEATS):
        for claim in latency_claims:
            _sync_device(PREFIX_DEVICE)
            if WINDOW_DEVICE != PREFIX_DEVICE:
                _sync_device(WINDOW_DEVICE)
            started = time.perf_counter()
            _latency_search(method, claim)
            _sync_device(PREFIX_DEVICE)
            if WINDOW_DEVICE != PREFIX_DEVICE:
                _sync_device(WINDOW_DEVICE)
            timings.append((time.perf_counter() - started) * 1000.0)
    latency_ms_by_method[method] = np.asarray(timings, dtype=np.float64)
    print(f"{method}: mean={np.mean(timings):.3f} ms, p95={np.percentile(timings, 95):.3f} ms")

# Keep the latest records for all five methods on this machine; older incomparable records are replaced.
efficiency_log_path = EFF_DIR / "efficiency_log.jsonl"
old_efficiency_rows = read_jsonl(efficiency_log_path) if efficiency_log_path.exists() else []
write_jsonl([row for row in old_efficiency_rows if row.get("method") not in latency_methods], efficiency_log_path)
latency_index_meta = {
    "bm25": (float(BM25_INDEX_TIME_S), 0, int(BM25_STORAGE_BYTES)),
    "prefix_minilm": (float(prefix_indexing_time_s), int(prefix_doc_emb.shape[0]), int(PREFIX_DOC_EMB_PATH.stat().st_size)),
    "window_minilm": (float(window_indexing_time_s), int(window_emb.shape[0]), int(WINDOW_EMB_PATH.stat().st_size)),
}
latency_index_meta["hybrid_prefix"] = (latency_index_meta["bm25"][0] + latency_index_meta["prefix_minilm"][0],
                                        latency_index_meta["prefix_minilm"][1],
                                        latency_index_meta["bm25"][2] + latency_index_meta["prefix_minilm"][2])
latency_index_meta["hybrid_window"] = (latency_index_meta["bm25"][0] + latency_index_meta["window_minilm"][0],
                                        latency_index_meta["window_minilm"][1],
                                        latency_index_meta["bm25"][2] + latency_index_meta["window_minilm"][2])

for method in latency_methods:
    measured = latency_ms_by_method[method]
    indexing_s, n_embeddings, storage_bytes = latency_index_meta[method]
    if method == "bm25":
        method_device = "CPU"
    elif method == "hybrid_prefix":
        method_device = f"CPU BM25 + {PREFIX_DEVICE}"
    elif method == "hybrid_window":
        method_device = f"CPU BM25 + {WINDOW_DEVICE}"
    elif method == "prefix_minilm":
        method_device = PREFIX_DEVICE
    else:
        method_device = WINDOW_DEVICE
    log_efficiency(method, indexing_time_s=indexing_s, n_embeddings=n_embeddings, storage_bytes=storage_bytes,
                   mean_latency_ms=float(measured.mean()), median_latency_ms=float(np.median(measured)),
                   p95_latency_ms=float(np.percentile(measured, 95)), max_latency_ms=float(measured.max()),
                   latency_queries=len(latency_claims), warmup_queries=len(latency_claims), n_repeats=LATENCY_REPEATS,
                   device=method_device, frozen_alpha=float(frozen_alphas[method]) if method.startswith("hybrid_") else None,
                   timing_boundary="query tokenisation + BM25 all-document scoring + MiniLM query encoding and all-document/window scoring as applicable + top-RUN_DEPTH candidate selection + min-max fusion and top-10 selection; model/index loading and file I/O excluded; warm; device synchronized",
                   shared_indexing_components=["bm25", "prefix_minilm" if method == "hybrid_prefix" else "window_minilm"] if method.startswith("hybrid_") else [method],
                   incremental_hybrid_storage_bytes=0 if method.startswith("hybrid_") else None)

efficiency_summary = pd.DataFrame([
    {"method": method, "mean_latency_ms": float(latency_ms_by_method[method].mean()),
     "p95_latency_ms": float(np.percentile(latency_ms_by_method[method], 95)),
     "indexing_time_s": latency_index_meta[method][0], "n_embeddings": latency_index_meta[method][1],
     "storage_bytes": latency_index_meta[method][2], "n_queries": len(latency_claims), "n_repeats": LATENCY_REPEATS}
    for method in latency_methods])
display(efficiency_summary)
print("Latency records updated for all five methods using one current-machine protocol; hybrid indexing/storage totals include both shared component indexes.")

---
# 8 · Final evaluation  *(Member 1)*

One evaluation script for all five configurations on the **300 untouched test queries**.

**Inputs (from Checkpoint A / disk)**
- `load_run(method, "test")` for every method in `METHODS`, `QRELS["test"]`
- `results/efficiency/efficiency_log.jsonl`

**Required outputs**
- `results/metrics/per_query__test.parquet`: `method, query_id, recall@10, ndcg@10` (Member 6 bootstraps from this)
- `results/metrics/summary__test.csv`: mean Recall@10, nDCG@10 per method
- `results/metrics/efficiency_summary.csv`: indexing time, embeddings, storage, mean/p95 latency per method

**Notes & checklist**
- [✅] Recall@10 = |relevant ∩ top-10| / |relevant|, averaged over every qrel query.
- [✅] nDCG@10 uses binary gains, log2 discount and the ideal DCG implied by each query's qrels.
- [✅] Cross-check with `pytrec_eval` when installed; otherwise compare the aggregate with the independent Section 3 metric helper.
- [✅] Every query in `QRELS["test"]` is scored; a relevant doc missing from a run is a miss, never a dropped query.

In [ ]:
# 8.1 Binary-gain metrics, one row per qrel query
def per_query_metrics(run, qrels, k=K):
    required = {"query_id", "doc_id", "rank"}
    if not required.issubset(run.columns):
        raise ValueError(f"run is missing columns: {sorted(required - set(run.columns))}")
    top = (run[run["rank"] <= k].sort_values(["query_id", "rank"])
           .groupby("query_id")["doc_id"].apply(list).to_dict())
    rows = []
    for qid, relevant in qrels.items():
        ranked = top.get(str(qid), [])
        hits = np.fromiter((float(doc in relevant) for doc in ranked), dtype=np.float64, count=len(ranked))
        recall = float(hits.sum() / len(relevant)) if relevant else 0.0
        discounts = 1.0 / np.log2(np.arange(2, len(hits) + 2))
        dcg = float(np.dot(hits, discounts))
        ideal_n = min(len(relevant), k)
        idcg = float(np.sum(1.0 / np.log2(np.arange(2, ideal_n + 2))))
        rows.append({"query_id": str(qid), f"recall@{k}": recall, f"ndcg@{k}": dcg / idcg if idcg else 0.0})
    return pd.DataFrame(rows)

def _cross_check_metrics(method, run, qrels, metrics, k=K):
    try:
        import pytrec_eval
    except ImportError:
        independent = quick_metrics(run, qrels, k=k)
        np.testing.assert_allclose(metrics[f"recall@{k}"].mean(), independent[f"recall@{k}"], atol=1e-12)
        np.testing.assert_allclose(metrics[f"ndcg@{k}"].mean(), independent[f"ndcg@{k}"], atol=1e-12)
        print(f"{method}: cross-checked against independent quick_metrics (pytrec_eval not installed)")
        return
    trec_qrels = {str(q): {str(d): int(rel) for d, rel in docs.items()} for q, docs in qrels.items()}
    trec_run = {}
    for qid, rows in run.groupby("query_id"):
        trec_run[str(qid)] = dict(zip(rows["doc_id"].astype(str), rows["score"].astype(float)))
    evaluator = pytrec_eval.RelevanceEvaluator(trec_qrels, {f"recall.{k}", f"ndcg_cut.{k}"})
    checked = evaluator.evaluate(trec_run)
    recall = np.mean([checked.get(str(q), {}).get(f"recall_{k}", 0.0) for q in qrels])
    ndcg = np.mean([checked.get(str(q), {}).get(f"ndcg_cut_{k}", 0.0) for q in qrels])
    np.testing.assert_allclose(metrics[f"recall@{k}"].mean(), recall, atol=1e-6)
    np.testing.assert_allclose(metrics[f"ndcg@{k}"].mean(), ndcg, atol=1e-6)
    print(f"{method}: pytrec_eval cross-check passed (absolute tolerance 1e-6)")

print("Per-query evaluation helpers ready")

In [ ]:
# Score every test qrel query for every method, including missing run rows as misses
per_query_parts = []
test_query_ids = set(QRELS["test"])
for method in METHODS:
    test_run = load_run(method, "test").copy()
    test_run["query_id"] = test_run["query_id"].astype(str)
    extra_qids = set(test_run["query_id"]) - test_query_ids
    if extra_qids:
        raise ValueError(f"{method}/test contains {len(extra_qids)} unexpected query IDs")
    metrics = per_query_metrics(test_run, QRELS["test"], k=K)
    metrics.insert(0, "method", method)
    per_query_parts.append(metrics)
    print(method, quick_metrics(test_run, QRELS["test"], k=K))

per_query_test = pd.concat(per_query_parts, ignore_index=True)
assert len(per_query_test) == len(METHODS) * len(QRELS["test"])
per_query_test.to_parquet(METRICS_DIR / "per_query__test.parquet", index=False)
summary_test = (per_query_test.groupby("method", sort=False)
                .agg(**{f"recall@{K}": (f"recall@{K}", "mean"), f"ndcg@{K}": (f"ndcg@{K}", "mean"), "n_queries": ("query_id", "nunique")})
                .reindex(METHODS).reset_index())
summary_test.to_csv(METRICS_DIR / "summary__test.csv", index=False)
_cross_check_metrics("bm25", load_run("bm25", "test"), QRELS["test"], per_query_test.query("method == 'bm25'"), k=K)
display(summary_test)
print(f"Saved {len(per_query_test):,} per-query rows across {len(METHODS)} methods.")

In [ ]:
# Latest available timing/index record per method; unavailable timings remain explicit NaN
efficiency_path = EFF_DIR / "efficiency_log.jsonl"
efficiency_rows = read_jsonl(efficiency_path) if efficiency_path.exists() else []
efficiency_frame = pd.DataFrame(efficiency_rows)
efficiency_fields = ["indexing_time_s", "n_embeddings", "storage_bytes", "mean_latency_ms", "p95_latency_ms"]
if not efficiency_frame.empty:
    efficiency_frame["timestamp_utc"] = pd.to_datetime(efficiency_frame.get("timestamp_utc"), utc=True, errors="coerce")

efficiency_summary_rows = []
for method in METHODS:
    records = efficiency_frame[efficiency_frame["method"] == method] if not efficiency_frame.empty and "method" in efficiency_frame else pd.DataFrame()
    record = records.sort_values("timestamp_utc").iloc[-1].to_dict() if not records.empty else {}
    efficiency_summary_rows.append({"method": method, **{field: record.get(field, np.nan) for field in efficiency_fields},
                                    "efficiency_timestamp_utc": record.get("timestamp_utc", pd.NaT),
                                    "hardware": record.get("hardware", ""),
                                    "efficiency_available": bool(record)})
efficiency_summary = pd.DataFrame(efficiency_summary_rows)
efficiency_summary.to_csv(METRICS_DIR / "efficiency_summary.csv", index=False)
display(efficiency_summary)
if not efficiency_summary["efficiency_available"].all():
    print("No direct efficiency record for: " + ", ".join(efficiency_summary.loc[~efficiency_summary.efficiency_available, "method"]) + ". Values are intentionally left blank, not inferred.")

---
# 9 · Statistical comparison, results table, charts  *(Member 6)*

Compare all methods, with uncertainty.

**Inputs (from Checkpoint A / disk)**
- `results/metrics/per_query__test.parquet`, `efficiency_summary.csv`, `diagnostic_records.parquet` (6.3)

**Required outputs**
- Paired bootstrap 95% CIs for metric **differences** (e.g. window − prefix, hybrid_window − hybrid_prefix, each vs BM25) → `results/metrics/bootstrap_ci.csv`
- Final results table (quality + efficiency) → `results/metrics/final_table.csv`
- Charts in `results/figures/` (metric bars with CIs, accuracy–latency trade-off, diagnostic by visibility group)

**Notes & checklist**
- [✅] Resample queries with replacement using the same paired query indices (10,000 draws, `SEED`).
- [✅] Report subgroup sizes alongside visibility-group hit rates.

In [ ]:
# 9.1 Paired bootstrap confidence intervals for pre-specified comparisons
N_BOOTSTRAP = 10_000
bootstrap_rng = np.random.default_rng(SEED)
query_ids = sorted(QRELS["test"])
per_query_index = per_query_test.set_index(["method", "query_id"])
metric_names = [f"recall@{K}", f"ndcg@{K}"]
metric_values = {
    (method, metric): per_query_index.loc[method].reindex(query_ids)[metric].to_numpy(dtype=np.float64)
    for method in METHODS for metric in metric_names
}
if any(not np.isfinite(values).all() for values in metric_values.values()):
    raise ValueError("Per-query evaluation contains missing or non-finite metric values")
bootstrap_indices = bootstrap_rng.integers(0, len(query_ids), size=(N_BOOTSTRAP, len(query_ids)))

comparisons = [
    ("prefix_minilm", "bm25"), ("window_minilm", "bm25"),
    ("hybrid_prefix", "bm25"), ("hybrid_window", "bm25"),
    ("window_minilm", "prefix_minilm"), ("hybrid_window", "hybrid_prefix"),
]
bootstrap_rows = []
for method_a, method_b in comparisons:
    for metric in metric_names:
        paired_diff = metric_values[(method_a, metric)] - metric_values[(method_b, metric)]
        sampled_means = paired_diff[bootstrap_indices].mean(axis=1)
        bootstrap_rows.append({
            "method_a": method_a, "method_b": method_b, "metric": metric,
            "mean_difference": float(paired_diff.mean()),
            "ci95_low": float(np.quantile(sampled_means, 0.025)),
            "ci95_high": float(np.quantile(sampled_means, 0.975)),
            "queries_a_better": int(np.sum(paired_diff > 0)),
            "queries_tied": int(np.sum(paired_diff == 0)),
            "queries_b_better": int(np.sum(paired_diff < 0)),
            "n_queries": len(query_ids), "n_bootstrap": N_BOOTSTRAP,
        })
bootstrap_ci = pd.DataFrame(bootstrap_rows)
bootstrap_ci.to_csv(METRICS_DIR / "bootstrap_ci.csv", index=False)

method_ci_rows = []
for method in METHODS:
    for metric in metric_names:
        values = metric_values[(method, metric)]
        sampled_means = values[bootstrap_indices].mean(axis=1)
        method_ci_rows.append({"method": method, "metric": metric, "mean": float(values.mean()),
                               "ci95_low": float(np.quantile(sampled_means, 0.025)),
                               "ci95_high": float(np.quantile(sampled_means, 0.975)), "n_queries": len(values)})
method_metric_ci = pd.DataFrame(method_ci_rows)
display(bootstrap_ci)
print(f"Saved paired difference intervals using {N_BOOTSTRAP:,} query-level resamples.")

In [ ]:
# 9.2 Merge test quality, measured efficiency and train-selected alpha
final_table = summary_test.merge(efficiency_summary, on="method", how="left", validate="one_to_one")
with open(RESULTS_DIR / "frozen_alphas.json", encoding="utf-8") as f:
    _frozen_alpha_summary = json.load(f)
final_table["alpha"] = final_table["method"].map({m: _frozen_alpha_summary.get(m, np.nan) for m in ["hybrid_prefix", "hybrid_window"]})
final_table["alpha_selected_by"] = final_table["method"].map(
    {"hybrid_prefix": _frozen_alpha_summary.get("selected_by", ""), "hybrid_window": _frozen_alpha_summary.get("selected_by", "")}).fillna("")
final_table.to_csv(METRICS_DIR / "final_table.csv", index=False)
display(final_table)
print("Efficiency fields are blank when no direct timing was recorded for a configuration; no estimates are substituted.")

In [ ]:
# 9.3 Test quality intervals, measured latency trade-off and evidence visibility
method_order = [method for method in METHODS if method in set(method_metric_ci["method"])]
fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
for ax, metric, title in zip(axes, metric_names, [f"Recall@{K}", f"nDCG@{K}"]):
    rows = method_metric_ci.query("metric == @metric").set_index("method").reindex(method_order)
    means = rows["mean"].to_numpy()
    errors = np.vstack([means - rows["ci95_low"].to_numpy(), rows["ci95_high"].to_numpy() - means])
    ax.bar(method_order, means, yerr=errors, capsize=4, color="steelblue")
    ax.set_title(title + " on test (95% query-bootstrap CI)")
    ax.set_ylim(0, 1)
    ax.tick_params(axis="x", rotation=35)
    ax.grid(axis="y", alpha=0.25)
fig.savefig(FIG_DIR / "test_metrics_with_ci.png", dpi=180, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(8, 5), constrained_layout=True)
tradeoff = final_table.dropna(subset=["mean_latency_ms", f"ndcg@{K}"])
for _, row in tradeoff.iterrows():
    ax.scatter(row["mean_latency_ms"], row[f"ndcg@{K}"], s=65)
    ax.annotate(row["method"], (row["mean_latency_ms"], row[f"ndcg@{K}"]), xytext=(5, 5), textcoords="offset points")
ax.set_xlabel("Mean query latency (ms; recorded method timing)")
ax.set_ylabel(f"Test nDCG@{K}")
ax.set_title("Accuracy–latency trade-off (methods with measured latency)")
ax.grid(alpha=0.25)
if tradeoff.empty:
    ax.text(0.5, 0.5, "No methods have a recorded latency", ha="center", va="center", transform=ax.transAxes)
fig.savefig(FIG_DIR / "accuracy_latency_tradeoff.png", dpi=180, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(9, 5), constrained_layout=True)
visible = visibility_summary[visibility_summary["visibility"].astype(str).isin(["complete", "partial", "absent"])]
if visible.empty:
    ax.text(0.5, 0.5, "No evidence-bearing test pairs for visibility analysis", ha="center", va="center", transform=ax.transAxes)
else:
    plot_data = visible.pivot(index="visibility", columns="method", values="hit_rate_at_10").reindex(["complete", "partial", "absent"])
    plot_data = plot_data.reindex(columns=METHODS)
    plot_data.plot(kind="bar", ax=ax)
    size_labels = visible.groupby("visibility", observed=True)["n_pairs"].max()
    ax.set_xticklabels([f"{v}\n(n={int(size_labels.get(v, 0))})" for v in plot_data.index], rotation=0)
ax.set_ylabel(f"Relevant-document hit rate @ {K}")
ax.set_title("Test retrieval by prefix evidence visibility")
ax.grid(axis="y", alpha=0.25)
fig.savefig(FIG_DIR / "visibility_recovery.png", dpi=180, bbox_inches="tight")
plt.show()
print("Saved test_metrics_with_ci.png, accuracy_latency_tradeoff.png and visibility_recovery.png")

---
# 10 · Demo export and reproducibility log  *(Member 6 / Member 1)*

Prepare what the Streamlit demo (`demo/app.py`) reads, and record what produced the results.

**Inputs (from Checkpoint A / disk)**
- Everything in `data/processed/` and `results/`

**Required outputs**
- Demo reads processed files + test runs directly (no extra export needed unless the demo requires live querying; then cache models/embeddings under `cache/`)
- `results/repro_log.json`: config snapshot, environment, file hashes of runs, check log

**Notes & checklist**
- [✅] Re-run the notebook top-to-bottom from a clean clone (done 2026-11-08, three runs: all quality metrics, α sweep, bootstrap CIs and diagnostics reproduced exactly; latency varies with hardware).
- [✅] `final_table.csv` is generated directly from the saved test evaluation and measured efficiency records.

In [ ]:
# 10.1 Validate files consumed by the Streamlit demo (all IDs remain strings)
demo_required = {
    "corpus": PROC_DIR / "corpus.parquet",
    "queries": PROC_DIR / "queries.parquet",
    "qrel_pairs": PROC_DIR / "qrel_pairs.parquet",
    "evidence_sentences": PROC_DIR / "evidence_sentences.parquet",
    "frozen_alphas": RESULTS_DIR / "frozen_alphas.json",
}
demo_required.update({f"{method}__test": run_path(method, "test") for method in METHODS})
missing_demo_files = [path.relative_to(ROOT).as_posix() for path in demo_required.values() if not path.is_file()]
if missing_demo_files:
    raise FileNotFoundError("Demo inputs are missing: " + ", ".join(missing_demo_files))
demo_queries = pd.read_parquet(demo_required["queries"], columns=["query_id", "split"])
demo_corpus = pd.read_parquet(demo_required["corpus"], columns=["doc_id"])
demo_test_queries = set(demo_queries.loc[demo_queries["split"] == "test", "query_id"].astype(str))
demo_corpus_ids = set(demo_corpus["doc_id"].astype(str))
for method in METHODS:
    demo_run = load_run(method, "test")
    if set(demo_run["query_id"].astype(str)) != demo_test_queries:
        raise ValueError(f"{method}/test query IDs do not match the processed test split")
    unknown_docs = set(demo_run["doc_id"].astype(str)) - demo_corpus_ids
    if unknown_docs:
        raise ValueError(f"{method}/test contains {len(unknown_docs)} doc IDs outside the corpus")
demo_manifest = {"corpus_documents": len(demo_corpus_ids), "test_queries": len(demo_test_queries),
                 "methods": METHODS, "inputs": {name: path.relative_to(ROOT).as_posix() for name, path in demo_required.items()},
                 "note": "The Streamlit app reads Parquet inputs directly; no model download is needed for saved-run browsing."}
(RESULTS_DIR / "demo_manifest.json").write_text(json.dumps(demo_manifest, indent=2), encoding="utf-8")
print(json.dumps(demo_manifest, indent=2))

In [ ]:
# 10.2 Hash inputs/outputs and record the checks available in this notebook run
repro_paths = [
    *(run_path(method, split) for method in METHODS for split in SPLITS),
    RESULTS_DIR / "frozen_alphas.json",
    METRICS_DIR / "alpha_sweep.parquet",
    EFF_DIR / "efficiency_log.jsonl",
    METRICS_DIR / "per_query__test.parquet",
    METRICS_DIR / "summary__test.csv",
    METRICS_DIR / "efficiency_summary.csv",
    METRICS_DIR / "bootstrap_ci.csv",
    METRICS_DIR / "final_table.csv",
    METRICS_DIR / "diagnostic_records.parquet",
    RESULTS_DIR / "demo_manifest.json",
    FIG_DIR / "test_metrics_with_ci.png",
    FIG_DIR / "accuracy_latency_tradeoff.png",
    FIG_DIR / "visibility_recovery.png",
]
repro_files = {}
for path in repro_paths:
    if path.is_file():
        repro_files[path.relative_to(ROOT).as_posix()] = {"sha256": sha256_file(path), "size_bytes": path.stat().st_size}
    else:
        raise FileNotFoundError(f"Expected reproducibility artifact is missing: {path.relative_to(ROOT)}")
if "CHECK_LOG" in globals() and CHECK_LOG:
    reproducibility_checks = CHECK_LOG
else:
    manifest_path = PROC_DIR / "manifest.json"
    reproducibility_checks = json.loads(manifest_path.read_text(encoding="utf-8")).get("checks", []) if manifest_path.exists() else []
repro_log = {
    "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "config": CONFIG,
    "environment": ENV,
    "split_map_beir_to_original": json.loads((PROC_DIR / "manifest.json").read_text(encoding="utf-8")).get("split_map_beir_to_original", {}),
    "files": repro_files,
    "checks": reproducibility_checks,
    "release_note": ("Clean-clone top-to-bottom reruns reproduce every quality metric exactly; "
                     "latency and indexing times depend on the hardware recorded in `environment`."),
}
(RESULTS_DIR / "repro_log.json").write_text(json.dumps(repro_log, indent=2, default=str), encoding="utf-8")
print(f"Wrote {RESULTS_DIR / 'repro_log.json'} with {len(repro_files)} SHA-256 entries and {len(reproducibility_checks)} recorded checks.")